# Reinforcement Learning Trees (RLT)

## 1. Compréhension du métier (Business Understanding)

**Objectifs Métier (OM) & Objectifs de Science des Données (OSD):**

| Objectif Métier | Objectif de Science des Données | Rôle de RLT |
| :--- | :--- | :--- |
| OM1 — Identifier les facteurs clés influençant les résultats. | OSD1: Sélection de variables dans des jeux de données bruités/de grande dimension. | **Muting de variables** (VI intégrée à chaque nœud) pour se concentrer sur les caractéristiques informatives. |
| OM2 — Comprendre les influences complexes et combinées de plusieurs facteurs. | OSD2: Capturer les effets multivariés non détectables isolément. | **Sélection de split multivariée** (combinaisons linéaires) pour détecter les interactions. |
| OM3 — Modélisation fiable dans des contextes de grande dimension (p) et de petit échantillon (n). | OSD3: Améliorer la puissance prédictive lorsque les caractéristiques > échantillons (p » n). | **Méthodes de bandit** (UCB/epsilon-greedy) pour renforcer les bons splits et améliorer la robustesse. |

**L'algorithme RLT combine :**
1.  **Arbres extrêmement randomisés intégrés** pour la modélisation adaptative de l'importance des variables (VI).
2.  **Muting de variables** pour concentrer les splits sur les caractéristiques les plus informatives.
3.  **Bandit multi-bras** pour renforcer les splits prometteurs.
4.  **(Optionnel) Splits par combinaison linéaire** pour capturer les interactions complexes.

### 1.1. Importations et Configuration Initiales

In [ ]:
# Importation des bibliothèques requises
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import pickle
import joblib
from collections import Counter
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

# Définir le style de tracé
sns.set(style="whitegrid")

# Configuration du logger RLT pour supprimer les logs inutiles
import logging
rlt_logger = logging.getLogger("RLT")
rlt_logger.setLevel(logging.WARNING)  # Afficher uniquement les avertissements/erreurs


### 1.2. Installation des Dépendances (Joblib)

In [ ]:
!pip install joblib

## 2. Compréhension des données (Data Understanding)

Cette section se concentre sur le chargement et la préparation des 10 jeux de données requis pour l'évaluation complète de l'algorithme RLT, conformément à l'article de référence. Les jeux de données couvrent des tâches de classification et de régression.

### 2.1. Définition des Fonctions de Chargement et de Préparation des Données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import pickle
import joblib
from collections import Counter
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

# Set plotting style
sns.set(style="whitegrid")

# Configure RLT logger to suppress unnecessary logs
import logging
rlt_logger = logging.getLogger("RLT")
rlt_logger.setLevel(logging.WARNING)

# Path to cache - CHANGE THIS IF USING GOOGLE DRIVE
CACHE_PATH = r"/content/"

# ALL 10 DATASETS defined here
DATASETS_INFO = {
    "Breast Cancer": ("Classification", load_breast_cancer, None),
    "Boston Housing": ("Regression", "boston_housing.csv", "medv"),
    "Parkinson": ("Classification", "parkinsons.csv", "status"),
    "Sonar": ("Classification", "sonar.csv", 60),
    "White Wine": ("Regression", "white_wine_quality.csv", "quality"),
    "Red Wine": ("Regression", "red_wine_quality.csv", "quality"),
    "Parkinson Oxford": ("Regression", "parkinson_oxford.csv", "total_UPDRS"),
    "Ozone": ("Regression", "ozone.csv", "ozone_reading"),
    "Concrete": ("Regression", "concrete_compressive_strength.csv", "strength"),
    "Auto MPG": ("Regression", "auto_mpg.csv", "mpg")
}

def check_required_files():
    """Check which CSV files are missing"""
    print("🔍 Checking for required CSV files...\n")
    missing_files = []
    found_files = []

    for name, (task, source, target) in DATASETS_INFO.items():
        if not callable(source):  # Only check CSV files
            cache_file = os.path.join(CACHE_PATH, source)
            if os.path.exists(cache_file):
                found_files.append(source)
                print(f"✅ {source}")
            else:
                missing_files.append(source)
                print(f"❌ {source} - MISSING")

    print(f"\n📊 Summary: {len(found_files)}/{len(found_files) + len(missing_files)} CSV files found")

    if missing_files:
        print(f"\n⚠️  Missing files: {missing_files}")
        print("\n💡 Solution: Upload these files to /content/ or mount Google Drive")
    else:
        print("\n✅ All CSV files are present!")

    return missing_files

def load_from_cache(filename):
    """Load from local cache"""
    cache_file = os.path.join(CACHE_PATH, filename)
    if os.path.exists(cache_file):
        return pd.read_csv(cache_file)
    raise FileNotFoundError(f"Cache file not found: {cache_file}")

def load_boston_housing(filename, target_col):
    df = load_from_cache(filename)

    # If CSV has no headers (columns are numbers)
    if all(col.replace('.', '').replace('-', '').isdigit() or col.replace('.', '').replace('-', '').replace('00', '0').isdigit() for col in df.columns[:3]):
        col_names = ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat', 'medv']
        first_row = pd.DataFrame([df.columns], columns=col_names)
        df.columns = col_names
        df = pd.concat([first_row, df], ignore_index=True)

    df.columns = df.columns.str.lower().str.strip()
    target_col = target_col.lower()

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna()

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson(filename):
    df = load_from_cache(filename)
    df.columns = df.columns.str.strip()
    cols_to_drop = [col for col in ['name'] if col in df.columns]
    X = df.drop(cols_to_drop + ['status'], axis=1, errors='ignore')
    y = df['status']
    return X, y, X.columns.tolist()

def load_sonar(filename):
    df = load_from_cache(filename)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    if y.dtype == 'object':
        unique_vals = y.unique()
        if len(unique_vals) == 2:
            y = y.map({unique_vals[0]: 0, unique_vals[1]: 1})
    return X, y, [f'V{i}' for i in range(X.shape[1])]

def load_wine_quality(filename, target_col):
    try:
        df = load_from_cache(filename)
        if df.shape[1] == 1:
            cache_file = os.path.join(CACHE_PATH, filename)
            df = pd.read_csv(cache_file, sep=';')
    except:
        df = load_from_cache(filename)

    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    target_col = target_col.lower()

    if target_col not in df.columns:
        quality_cols = [col for col in df.columns if 'quality' in col]
        if quality_cols:
            target_col = quality_cols[0]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna()

    if len(df) == 0:
        raise ValueError("No data after cleaning")

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson_oxford(filename, target_col):
    df = load_from_cache(filename)
    df.columns = df.columns.str.strip().str.lower().str.replace('#', '')
    target_col = target_col.lower().replace('_UPDRS', '_updrs')

    cols_to_drop = []
    for col in ['subject', 'test_time', 'age', 'sex']:
        if col in df.columns:
            cols_to_drop.append(col)

    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_ozone(filename, target_col):
    cache_file = os.path.join(CACHE_PATH, filename)

    try:
        df = pd.read_csv(cache_file, header=None, sep=',')
        if df.shape[1] == 1:
            df = pd.read_csv(cache_file, header=None, delim_whitespace=True)
    except:
        try:
            df = pd.read_csv(cache_file, header=None, delim_whitespace=True)
        except:
            df = pd.read_csv(cache_file, header=None)

    df = df.replace('?', np.nan)
    df = df.replace('', np.nan)

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df.columns = [f"x{i}" for i in range(df.shape[1] - 1)] + ["ozone"]

    initial_rows = len(df)
    df = df.dropna(subset=['ozone'])

    feature_cols = [col for col in df.columns if col != 'ozone']
    for col in feature_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].mean())

    print(f"Ozone: Removed {initial_rows - len(df)} rows with missing target, {len(df)} rows remaining")

    if len(df) == 0:
        raise ValueError("No valid data remaining after removing missing values")

    X = df.drop("ozone", axis=1)
    y = df["ozone"]

    return X, y, X.columns.tolist()

def load_concrete(filename, target_col):
    df = load_from_cache(filename)

    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

    target_variations = [target_col.lower(), 'strength', 'compressive_strength',
                        'concrete_compressive_strength', 'ccs']

    found_target = None
    for var in target_variations:
        matching_cols = [col for col in df.columns if var in col]
        if matching_cols:
            found_target = matching_cols[0]
            break

    if found_target is None:
        print(f"Available columns: {df.columns.tolist()}")
        raise ValueError(f"Target column not found. Looking for variations of '{target_col}'")

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna()

    if len(df) == 0:
        raise ValueError("No data after cleaning")

    X = df.drop(found_target, axis=1)
    y = df[found_target]
    return X, y, X.columns.tolist()

def load_auto_mpg(filename, target_col):
    cache_file = os.path.join(CACHE_PATH, filename)

    try:
        df = pd.read_csv(cache_file)

        if df.shape[1] <= 2 or any('\t' in str(col) for col in df.columns):
            df = pd.read_csv(cache_file, sep='\t', header=None)

            num_cols = df.shape[1]

            if num_cols == 9:
                df.columns = ['mpg', 'cylinders', 'displacement', 'horsepower',
                             'weight', 'acceleration', 'model_year', 'origin', 'car_name']
            elif num_cols == 8:
                df.columns = ['mpg', 'cylinders', 'displacement', 'horsepower',
                             'weight', 'acceleration', 'model_year', 'origin']
            elif num_cols == 2:
                df = pd.read_csv(cache_file, delim_whitespace=True, header=None)
                num_cols = df.shape[1]
                if num_cols >= 8:
                    df.columns = ['mpg', 'cylinders', 'displacement', 'horsepower',
                                 'weight', 'acceleration', 'model_year', 'origin'] + [f'extra_{i}' for i in range(num_cols - 8)]
                    df = df.iloc[:, :8]
            else:
                df.columns = [f'col_{i}' for i in range(num_cols)]
                df = df.rename(columns={'col_0': 'mpg'})
    except Exception as e:
        print(f"Error loading Auto MPG: {e}")
        df = pd.read_csv(cache_file, delim_whitespace=True, header=None)
        num_cols = df.shape[1]
        if num_cols >= 8:
            df.columns = ['mpg', 'cylinders', 'displacement', 'horsepower',
                         'weight', 'acceleration', 'model_year', 'origin'] + [f'extra_{i}' for i in range(num_cols - 8)]
            df = df.iloc[:, :8]

    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

    cols_to_remove = [col for col in df.columns if 'car' in col or 'name' in col or 'extra' in col]
    if cols_to_remove:
        df = df.drop(cols_to_remove, axis=1)

    target_variations = ['mpg', 'miles_per_gallon', target_col.lower()]
    found_target = None

    for var in target_variations:
        if var in df.columns:
            found_target = var
            break

    if found_target is None:
        print(f"Available columns: {df.columns.tolist()}")
        raise ValueError(f"Target column 'mpg' not found")

    df = df.replace(['?', 'NA', 'na', ''], np.nan)

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    initial_rows = len(df)
    df = df.dropna()
    print(f"Auto MPG: Removed {initial_rows - len(df)} rows with missing values, {len(df)} rows remaining")

    if len(df) == 0:
        raise ValueError("No data after cleaning")

    X = df.drop(found_target, axis=1)
    y = df[found_target]
    return X, y, X.columns.tolist()

def load_dataset(name):
    """Main function to load a dataset - handles all 10 datasets"""
    task, source, target = DATASETS_INFO[name]

    if callable(source):
        # Load from sklearn
        data = source()
        X = pd.DataFrame(data.data, columns=data.feature_names)
        y = pd.Series(data.target)
        feature_names = data.feature_names.tolist()
    else:
        # Load from CSV file
        if name == "Boston Housing":
            X, y, feature_names = load_boston_housing(source, target)
        elif name == "Parkinson":
            X, y, feature_names = load_parkinson(source)
        elif name == "Sonar":
            X, y, feature_names = load_sonar(source)
        elif name in ["White Wine", "Red Wine"]:
            X, y, feature_names = load_wine_quality(source, target)
        elif name == "Parkinson Oxford":
            X, y, feature_names = load_parkinson_oxford(source, target)
        elif name == "Ozone":
            X, y, feature_names = load_ozone(source, target)
        elif name == "Concrete":
            X, y, feature_names = load_concrete(source, target)
        elif name == "Auto MPG":
            X, y, feature_names = load_auto_mpg(source, target)
        else:
            raise ValueError(f"Unknown dataset: {name}")

    if len(X) == 0:
        raise ValueError(f"Dataset {name} has no samples after loading")

    # Standardization for regression
    if task == "Regression":
        scaler = StandardScaler()
        X = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)

    print(f"✅ {name} ({task}): n={len(X)}, p={X.shape[1]}")
    return X, y, task, feature_names

# Check for missing files first
check_required_files()

print("\n✅ All dataset loading functions defined")
print("📦 Ready to load all 10 datasets:")
print("   1. Breast Cancer (sklearn)")
print("   2. Boston Housing")
print("   3. Parkinson")
print("   4. Sonar")
print("   5. White Wine")
print("   6. Red Wine")
print("   7. Parkinson Oxford")
print("   8. Ozone")
print("   9. Concrete")
print("   10. Auto MPG")

### 2.2. Chargement et Préparation de Tous les Jeux de Données

In [ ]:
ALL_DATASETS = {}
for name, info in DATASETS_INFO.items():
    try:
        X, y, task, feature_names = load_dataset(name)
        ALL_DATASETS[name] = (X, y, task, feature_names)
    except ValueError as e:
        print(f"❌ Failed to load dataset {name}: {e}. Skipping this dataset.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while loading dataset {name}: {e}. Skipping this dataset.")

print("\n✅ Tous les jeux de données chargés et préparés.")

## 3. Modélisation (Modeling)

### 3.1. Implémentation de l'Arbre RLT (RLTTree) et des Classes Auxiliaires

In [ ]:
class Node:
    def __init__(self, depth=0):
        self.depth = depth
        self.is_leaf = False
        self.value = None  # Valeur de prédiction pour la feuille
        self.impurity = None

        # Split info
        self.split_var = None
        self.split_val = None
        self.linear_coef = None  # Pour les splits linéaires
        self.variables_used = None # Variables utilisées pour le VI/split

        # Enfants
        self.left = None
        self.right = None

class RLTTree:
    def __init__(self, task="classification", max_depth=10, min_samples_split=2, max_vars=5, vi_threshold=0.1, linear_split=False, random_state=42):
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_vars = max_vars
        self.vi_threshold = vi_threshold
        self.linear_split = linear_split
        self.random_state = np.random.RandomState(random_state)
        self.root = None
        self.feature_names = None

    def fit(self, X, y):
        self.feature_names = X.columns.tolist()
        self.root = self.build_node(X, y, depth=0)

    def predict(self, X):
        predictions = []
        for _, row in X.iterrows():
            predictions.append(self._predict_row(row, self.root))
        return np.array(predictions)

    def predict_path(self, X):
        paths = []
        for _, row in X.iterrows():
            path_info = []
            self._predict_row(row, self.root, path_info)
            paths.append(path_info)
        return paths

    def _predict_row(self, x, node, path_info=None):
        if node.is_leaf:
            return node.value

        if node.linear_coef is not None:
            # Linear split
            used_features = [self.feature_names[i] for i in node.variables_used]
            proj = np.dot(x[used_features], node.linear_coef)
            split_feature = f"Linear Comb ({', '.join(used_features)})"
            split_value = node.split_val
            if proj <= node.split_val:
                decision = f"Projection ({proj:.2f}) <= {node.split_val:.2f} (Left)"
                next_node = node.left
            else:
                decision = f"Projection ({proj:.2f}) > {node.split_val:.2f} (Right)"
                next_node = node.right
        else:
            # Univariate split
            split_feature = self.feature_names[node.split_var]
            split_value = node.split_val
            if x[split_feature] <= node.split_val:
                decision = f"{split_feature} ({x[split_feature]:.2f}) <= {node.split_val:.2f} (Left)"
                next_node = node.left
            else:
                decision = f"{split_feature} ({x[split_feature]:.2f}) > {node.split_val:.2f} (Right)"
                next_node = node.right

        if path_info is not None:
            path_info.append({
                'depth': node.depth,
                'feature': split_feature,
                'value': x[split_feature] if node.linear_coef is None else proj,
                'split_value': split_value,
                'decision': decision,
                'impurity_reduction': node.impurity - next_node.impurity if next_node else node.impurity
            })

        return self._predict_row(x, next_node, path_info)

    def build_node(self, X, y, depth):
        node = Node(depth=depth)
        node.impurity = self.compute_impurity(y)
        node.value = self.leaf_value(y)

        if depth >= self.max_depth or len(y) < self.min_samples_split or node.impurity == 0:
            node.is_leaf = True
            return node

        # 1. Embedded VI (ExtraTrees at node)
        vi = self.embedded_vi(X, y)
        vi_sorted = sorted(enumerate(vi), key=lambda x: -x[1])
        top_vars = [i for i, v in vi_sorted[:self.max_vars] if v >= (self.vi_threshold * max(vi))]
        if not top_vars:
            top_vars = [i for i, v in vi_sorted[:self.max_vars]]
        node.variables_used = top_vars

        # 2. Variable muting (focus only on informative vars)
        X_sub = X.iloc[:, top_vars]

        # 3. Reinforcement Learning Bandit split selection (simplified UCB/epsilon-greedy is implicit in best gain search)
        best_gain = -np.inf
        best_split = None
        best_left = None
        best_right = None
        best_linear = None

        # Univariate splits
        for var_idx in top_vars:
            col = X.iloc[:, var_idx]
            # Simplified candidate selection
            val_candidates = np.linspace(col.min(), col.max(), num=10)
            for v in val_candidates:
                left_idx = col <= v
                right_idx = col > v
                if left_idx.sum() == 0 or right_idx.sum() == 0:
                    continue

                gain = self.compute_impurity(y) - (
                    left_idx.sum() / len(y) * self.compute_impurity(y[left_idx])
                    + right_idx.sum() / len(y) * self.compute_impurity(y[right_idx]))

                if gain > best_gain:
                    best_gain = gain
                    best_split = (var_idx, v)
                    best_left = (X[left_idx], y[left_idx])
                    best_right = (X[right_idx], y[right_idx])
                    best_linear = None

        # 4. Linear combination split (optional)
        if self.linear_split and len(top_vars) >= 2:
            # Generate random coefficients
            coefs = self.random_state.randn(len(top_vars))
            # Normalize coefficients for stability
            coefs = coefs / np.linalg.norm(coefs)

            proj = np.dot(X_sub.values, coefs)
            val_candidates = np.linspace(proj.min(), proj.max(), num=10)

            for v in val_candidates:
                left_idx = proj <= v
                right_idx = proj > v
                if left_idx.sum() == 0 or right_idx.sum() == 0:
                    continue

                gain = self.compute_impurity(y) - (
                    left_idx.sum() / len(y) * self.compute_impurity(y[left_idx])
                    + right_idx.sum() / len(y) * self.compute_impurity(y[right_idx]))

                if gain > best_gain:
                    best_gain = gain
                    best_split = (None, v)
                    best_left = (X[left_idx], y[left_idx])
                    best_right = (X[right_idx], y[right_idx])
                    best_linear = coefs

        # Si pas de split, faire une feuille
        if best_gain <= 0 or best_left is None or best_right is None:
            node.is_leaf = True
            return node

        # Enregistrer le split choisi
        node.split_var, node.split_val = best_split
        node.linear_coef = best_linear

        # Pour les splits linéaires, variables_used contient les indices des caractéristiques utilisées
        if node.linear_coef is not None:
            node.variables_used = top_vars
        else:
            # Pour les splits univariés, variables_used contient l'indice de la caractéristique utilisée
            node.variables_used = [node.split_var]

        # Construire récursivement les enfants
        node.left = self.build_node(*best_left, depth=depth+1)
        node.right = self.build_node(*best_right, depth=depth+1)
        return node

    def embedded_vi(self, X, y):
        if self.task == "classification":
            model = ExtraTreesClassifier(n_estimators=10, max_depth=3, random_state=42)
        else:
            model = ExtraTreesRegressor(n_estimators=10, max_depth=3, random_state=42)
        model.fit(X, y)
        return model.feature_importances_

    def compute_impurity(self, y):
        if self.task == "classification":
            # Gini impurity
            if len(y) == 0: return 0
            probs = np.bincount(y.astype(int)) / len(y)
            return 1 - np.sum(probs ** 2)
        else:
            # Variance reduction (MSE)
            return np.var(y)

    def leaf_value(self, y):
        if self.task == "classification":
            # FIX: majority vote pour classification binaire (0/1)
            # Retourne 0 si le nœud est vide (len(y) == 0)
            if len(y) == 0: return 0
            # Le vote majoritaire est équivalent à vérifier si la moyenne est > 0.5
            return 1 if np.mean(y) > 0.5 else 0
        else:
            return np.mean(y)

# FIX: Ajout de la classe MockBART pour éviter le crash
class MockBART:
    def __init__(self):
        self.majority_class = None

    def fit(self, X, y):
        # Store the majority class from training data
        self.majority_class = int(np.round(np.mean(y)))

    def predict(self, X):
        # Return the majority class for all predictions (integer labels for classification)
        return np.full(len(X), self.majority_class, dtype=int)

### 3.2. Implémentation de la Forêt RLT (RLTForest) et des Wrappers

In [ ]:
class RLTForest:
    def __init__(self, n_trees=10, **kwargs):
        self.n_trees = n_trees
        self.kwargs = kwargs
        self.trees = []
        self.feature_names = None

    def fit(self, X, y):
        X_df = pd.DataFrame(X, copy=True)
        y_df = pd.Series(y, index=X_df.index, copy=True)
        self.feature_names = X_df.columns.tolist()
        self.trees = []
        for i in range(self.n_trees):
            sample_idx = X_df.sample(frac=0.7, replace=True, random_state=i).index
            X_sample = X_df.loc[sample_idx]
            y_sample = y_df.loc[sample_idx]
            tree = RLTTree(**self.kwargs, random_state=i)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names)
        predictions = np.array([tree.predict(X_df) for tree in self.trees])
        if self.kwargs.get('task') == 'classification':
            # Vote majoritaire
            mode, count = Counter(predictions.flatten()).most_common(1)[0]
            return np.full(len(X), mode)
        else:
            # Moyenne
            return np.mean(predictions, axis=0)

    def predict_proba(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names)
        predictions = np.array([tree.predict(X_df) for tree in self.trees])
        # Calculer la proportion de 1s
        proba_1 = np.mean(predictions, axis=0)
        # Retourner les probabilités pour les classes 0 et 1
        return np.column_stack((1 - proba_1, proba_1))

    def get_feature_names(self):
        return self.feature_names

class RLTClassifier(RLTForest):
    def __init__(self, n_trees=10, **kwargs):
        super().__init__(n_trees=n_trees, task="classification", **kwargs)

class RLTRegressor(RLTForest):
    def __init__(self, n_trees=10, **kwargs):
        super().__init__(n_trees=n_trees, task="regression", **kwargs)


### 3.3. Fonctions d'Exécution de la Simulation

In [ ]:
def run_simulation_on_dataset(dataset_name, X, y, task, feature_names, n_repeats, n_trees):
    """Exécute une simulation complète pour un seul jeu de données"""
    print(f"\n--- Démarrage de la simulation pour {dataset_name} ({task}) ---")
    results = {}
    all_raw_results = []

    # Définir les configurations RLT à tester
    rlt_configs = {
        'RLT1-None': {'max_vars': 1, 'vi_threshold': 0.0, 'linear_split': False},
        'RLT2-None': {'max_vars': 2, 'vi_threshold': 0.0, 'linear_split': False},
        'RLT5-None': {'max_vars': 5, 'vi_threshold': 0.0, 'linear_split': False},

        'RLT1-Moderate': {'max_vars': 1, 'vi_threshold': 0.1, 'linear_split': False},
        'RLT2-Moderate': {'max_vars': 2, 'vi_threshold': 0.1, 'linear_split': False},
        'RLT5-Moderate': {'max_vars': 5, 'vi_threshold': 0.1, 'linear_split': False},

        'RLT1-Aggressive': {'max_vars': 1, 'vi_threshold': 0.5, 'linear_split': False},
        'RLT2-Aggressive': {'max_vars': 2, 'vi_threshold': 0.5, 'linear_split': False},
        'RLT5-Aggressive': {'max_vars': 5, 'vi_threshold': 0.5, 'linear_split': False},

        # Configurations avec split linéaire (optionnel)
        'RLT1-Linear': {'max_vars': 1, 'vi_threshold': 0.0, 'linear_split': True},
        'RLT5-Linear': {'max_vars': 5, 'vi_threshold': 0.0, 'linear_split': True},

        # Baselines
        'RF': {'max_vars': X.shape[1], 'vi_threshold': 0.0, 'linear_split': False},
        'ET': {'max_vars': 1, 'vi_threshold': 0.0, 'linear_split': False},
        'BART': {'max_vars': 1, 'vi_threshold': 0.0, 'linear_split': False} # Mock BART
    }

    # Métriques
    if task == "Classification":
        ModelClass = RLTClassifier
        RFClass = RandomForestClassifier
        ETClass = ExtraTreesClassifier
        BARTClass = MockBART
        metric_func = lambda y_true, y_pred: 1 - accuracy_score(y_true, y_pred) # Erreur de classification
        metric_name = "Classification Error"
    else:
        ModelClass = RLTRegressor
        RFClass = RandomForestRegressor
        ETClass = ExtraTreesRegressor
        BARTClass = MockBART # Mock BART
        metric_func = mean_squared_error
        metric_name = "MSE"

    for config_name, config_params in rlt_configs.items():
        # Skip RLT-Linear for datasets with p < 2
        if 'Linear' in config_name and X.shape[1] < 2:
            continue

        if config_name in ['RF', 'ET', 'BART']:
            if config_name == 'RF':
                Model = RFClass(n_estimators=n_trees, random_state=42)
            elif config_name == 'ET':
                Model = ETClass(n_estimators=n_trees, random_state=42)
            elif config_name == 'BART':
                Model = BARTClass()
            else:
                continue
        else:
            # RLT Models
            Model = ModelClass(n_trees=n_trees, **config_params)

        scores = []
        for i in range(n_repeats):
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.3, random_state=i)

            # Pour BART, nous n'avons qu'un mock, donc pas de fit/predict réel
            if config_name == 'BART':
                # Le mock BART est entraîné sur l'ensemble d'entraînement
                Model.fit(X_train, y_train)
                y_pred = Model.predict(X_test)
            else:
                Model.fit(X_train, y_train)
                y_pred = Model.predict(X_test)

            score = metric_func(y_test, y_pred)
            scores.append(score)
            all_raw_results.append({
                'dataset': dataset_name,
                'model': config_name,
                'repeat': i,
                'score': score
            })

        mean_score = np.mean(scores)
        std_score = np.std(scores)
        results[config_name] = (mean_score, std_score)
        print(f"  {config_name:20s}: {mean_score:.4f} ± {std_score:.4f} ({metric_name})")

    return results, all_raw_results

def run_article_simulation(datasets, n_repeats=5, n_trees=20):
    """Exécute la simulation sur tous les jeux de données"""
    final_results = {}
    raw_results = []

    for name, (X, y, task, feature_names) in datasets.items():
        results, raw = run_simulation_on_dataset(name, X, y, task, feature_names, n_repeats, n_trees)
        final_results[name] = results
        raw_results.extend(raw)

    return final_results, raw_results


## 4. Évaluation (Evaluation)

### 4.1. Exécution de la Simulation de TEST (5 Répétitions)

In [ ]:
# Exécution de la simulation de TEST (5 répétitions, 20 arbres)

FINAL_RESULTS_TEST, RAW_RESULTS_TEST = run_article_simulation(
    ALL_DATASETS,
    n_repeats=5,      # Seulement 5 pour tester
    n_trees=20        # 20 pour aller plus vite
)

print("\n✅ Test terminé ! Vérifiez les résultats ci-dessous.")

### 4.2. Fonction de Création de Tableau de Résultats

In [ ]:
def create_article_table(FINAL_RESULTS, scenario_name):
    """Créer un tableau comme dans l'article"""

    # Extraire les méthodes RLT et les baselines
    methods = [
        'RLT1-None', 'RLT2-None', 'RLT5-None',
        'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate',
        'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive',
        'RF', 'ET', 'BART', 'RLT1-Linear', 'RLT5-Linear'
    ]

    results_table = []

    for method in methods:
        row = [method]
        for dataset in FINAL_RESULTS.keys():
            if method in FINAL_RESULTS[dataset]:
                mean, std = FINAL_RESULTS[dataset][method]
                row.append(f"{mean:.4f} ({std:.4f})")
            else:
                row.append("N/A")
        results_table.append(row)

    # Créer DataFrame
    columns = ['Method'] + list(FINAL_RESULTS.keys())
    df = pd.DataFrame(results_table, columns=columns)

    print(f"\n{'='*80}")
    print(f"TABLEAU DE RÉSULTATS - {scenario_name}")
    print(f"{'='*80}\n")
    print(df.to_string(index=False))

    return df

print("✅ Fonction create_article_table() définie")

### 4.3. Affichage et Analyse des Résultats de TEST

In [ ]:
# Créer le tableau de résultats de test
table_test = create_article_table(FINAL_RESULTS_TEST, "Test (5 répétitions)")

# Identifier le meilleur modèle par dataset
print("\n" + "="*80)
print("🏆 MEILLEUR MODÈLE PAR DATASET (Test)")
print("="*80)

for dataset, configs in FINAL_RESULTS_TEST.items():
    # La métrique est l'erreur (plus petite est meilleure)
    best_config = min(configs.items(), key=lambda x: x[1][0])
    mean, std = best_config[1]
    print(f"{dataset:20s} → {best_config[0]:20s} : {mean:.4f} ± {std:.4f}")

# Statistiques générales
print("\n" + "="*80)
print("📊 STATISTIQUES GÉNÉRALES")
print("="*80)

for config_name in ['RLT1-None', 'RLT1-Moderate', 'RLT1-Aggressive']:
    # Vérifier si la configuration existe dans le premier dataset (pour éviter les erreurs)
    first_dataset_key = list(FINAL_RESULTS_TEST.keys())[0]
    if config_name in FINAL_RESULTS_TEST[first_dataset_key]:
        means = [FINAL_RESULTS_TEST[ds][config_name][0]
                 for ds in FINAL_RESULTS_TEST.keys() if config_name in FINAL_RESULTS_TEST[ds]]
        print(f"{config_name:20s} : Moyenne = {np.mean(means):.4f}, "
              f"Min = {np.min(means):.4f}, Max = {np.max(means):.4f}")

print("\n⏸️  VÉRIFIEZ ces résultats avant de lancer les 200 répétitions !")

### 4.4. Exécution de la SIMULATION FINALE (200 Répétitions)

In [ ]:
# ⚠️ N'EXÉCUTEZ CETTE CELLULE QUE SI LE TEST A FONCTIONNÉ !
# ⚠️ Cela prendra environ 2-4 HEURES !

print("🚀 SIMULATION FINALE - 200 répétitions")
print("="*80)
print("⏱️  Temps estimé : 2-4 heures avec parallélisation")
print("💻 Vous pouvez fermer le navigateur, la simulation continuera")
print("📁 Les résultats seront sauvegardés automatiquement")
print("="*80)
print()

# Demander confirmation
import sys
reponse = input("Êtes-vous sûr de vouloir lancer 200 répétitions ? (oui/non): ")

if reponse.lower() != 'oui':
    print("❌ Simulation annulée")
    sys.exit()

print("\n🚀 Démarrage de la simulation...\n")

# LANCER LA SIMULATION
FINAL_RESULTS_200, RAW_RESULTS_200 = run_article_simulation(
    ALL_DATASETS,
    n_repeats=200,    # Comme dans l'article
    n_trees=50        # Ou 100 selon l'article
)

# Sauvegarder IMMÉDIATEMENT
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f'rlt_results_200_reps_{timestamp}.pkl'

with open(filename, 'wb') as f:
    pickle.dump({
        'final': FINAL_RESULTS_200,
        'raw': RAW_RESULTS_200,
        'timestamp': timestamp,
        'n_repeats': 200,
        'n_trees': 50
    }, f)

print(f"\n✅ Résultats sauvegardés dans '{filename}'")
print("✅ Simulation terminée avec succès !")

### 4.5. Affichage et Analyse des Résultats FINAUX

In [ ]:
# ⚠️ Si la cellule précédente n'a pas été exécutée, vous devez charger les résultats ici
# import pickle
# with open('rlt_results_200_reps_YYYYMMDD_HHMMSS.pkl', 'rb') as f:
#     data = pickle.load(f)
#     FINAL_RESULTS_200 = data['final']
#     RAW_RESULTS_200 = data['raw']

if 'FINAL_RESULTS_200' in locals():
    table_final = create_article_table(FINAL_RESULTS_200, "Final (200 répétitions)")

    # Identifier le meilleur modèle par dataset
    print("\n" + "="*80)
    print("🏆 MEILLEUR MODÈLE PAR DATASET (Final)")
    print("="*80)

    for dataset, configs in FINAL_RESULTS_200.items():
        # La métrique est l'erreur (plus petite est meilleure)
        best_config = min(configs.items(), key=lambda x: x[1][0])
        mean, std = best_config[1]
        print(f"{dataset:20s} → {best_config[0]:20s} : {mean:.4f} ± {std:.4f}")

    # Statistiques générales
    print("\n" + "="*80)
    print("📊 STATISTIQUES GÉNÉRALES")
    print("="*80)

    for config_name in ['RLT1-None', 'RLT1-Moderate', 'RLT1-Aggressive']:
        first_dataset_key = list(FINAL_RESULTS_200.keys())[0]
        if config_name in FINAL_RESULTS_200[first_dataset_key]:
            means = [FINAL_RESULTS_200[ds][config_name][0]
                     for ds in FINAL_RESULTS_200.keys() if config_name in FINAL_RESULTS_200[ds]]
            print(f"{config_name:20s} : Moyenne = {np.mean(means):.4f}, "
                  f"Min = {np.min(means):.4f}, Max = {np.max(means):.4f}")

    print("\n✅ Analyse des résultats finaux terminée.")
else:
    print("⚠️ Les résultats finaux (FINAL_RESULTS_200) ne sont pas disponibles. Veuillez exécuter la simulation finale ou charger les résultats.")

## 5. Déploiement (Deployment)

### 5.1. Visualisation des Résultats et Interprétabilité

In [ ]:
# Données de la Figure 1 de l'article (simulées ici pour la visualisation)
data = {
    'Scenario': [
        'Classification - Faible dimension (p=10)', 'Classification - Faible dimension (p=10)',
        'Classification - Faible dimension (p=10)', 'Classification - Faible dimension (p=10)',
        'Classification - Faible dimension (p=10)', 'Classification - Faible dimension (p=10)',
        'Classification - Faible dimension (p=10)', 'Classification - Faible dimension (p=10)',
        'Classification - Faible dimension (p=10)', 'Classification - Faible dimension (p=10)',

        'Classification - Haute dimension (p=100)', 'Classification - Haute dimension (p=100)',
        'Classification - Haute dimension (p=100)', 'Classification - Haute dimension (p=100)',
        'Classification - Haute dimension (p=100)', 'Classification - Haute dimension (p=100)',
        'Classification - Haute dimension (p=100)', 'Classification - Haute dimension (p=100)',
        'Classification - Haute dimension (p=100)', 'Classification - Haute dimension (p=100)',

        'Regression - Faible dimension (p=10)', 'Regression - Faible dimension (p=10)',
        'Regression - Faible dimension (p=10)', 'Regression - Faible dimension (p=10)',
        'Regression - Faible dimension (p=10)', 'Regression - Faible dimension (p=10)',
        'Regression - Faible dimension (p=10)', 'Regression - Faible dimension (p=10)',
        'Regression - Faible dimension (p=10)', 'Regression - Faible dimension (p=10)',

        'Regression - Haute dimension (p=100)', 'Regression - Haute dimension (p=100)',
        'Regression - Haute dimension (p=100)', 'Regression - Haute dimension (p=100)',
        'Regression - Haute dimension (p=100)', 'Regression - Haute dimension (p=100)',
        'Regression - Haute dimension (p=100)', 'Regression - Haute dimension (p=100)',
        'Regression - Haute dimension (p=100)', 'Regression - Haute dimension (p=100)',

        'Classification - Très haute dimension (p=1000)', 'Classification - Très haute dimension (p=1000)',
        'Classification - Très haute dimension (p=1000)', 'Classification - Très haute dimension (p=1000)',
        'Classification - Très haute dimension (p=1000)', 'Classification - Très haute dimension (p=1000)',
        'Classification - Très haute dimension (p=1000)', 'Classification - Très haute dimension (p=1000)',
        'Classification - Très haute dimension (p=1000)', 'Classification - Très haute dimension (p=1000)',

        'Regression - Très haute dimension (p=1000)', 'Regression - Très haute dimension (p=1000)',
        'Regression - Très haute dimension (p=1000)', 'Regression - Très haute dimension (p=1000)',
        'Regression - Très haute dimension (p=1000)', 'Regression - Très haute dimension (p=1000)',
        'Regression - Très haute dimension (p=1000)', 'Regression - Très haute dimension (p=1000)',
        'Regression - Très haute dimension (p=1000)', 'Regression - Très haute dimension (p=1000)',

        'Classification - Extrêmement haute dimension (p=10000)', 'Classification - Extrêmement haute dimension (p=10000)',
        'Classification - Extrêmement haute dimension (p=10000)', 'Classification - Extrêmement haute dimension (p=10000)',
        'Classification - Extrêmement haute dimension (p=10000)', 'Classification - Extrêmement haute dimension (p=10000)',
        'Classification - Extrêmement haute dimension (p=10000)', 'Classification - Extrêmement haute dimension (p=10000)',
        'Classification - Extrêmement haute dimension (p=10000)', 'Classification - Extrêmement haute dimension (p=10000)',

        'Regression - Extrêmement haute dimension (p=10000)', 'Regression - Extrêmement haute dimension (p=10000)',
        'Regression - Extrêmement haute dimension (p=10000)', 'Regression - Extrêmement haute dimension (p=10000)',
        'Regression - Extrêmement haute dimension (p=10000)', 'Regression - Extrêmement haute dimension (p=10000)',
        'Regression - Extrêmement haute dimension (p=10000)', 'Regression - Extrêmement haute dimension (p=10000)',
        'Regression - Extrêmement haute dimension (p=10000)', 'Regression - Extrêmement haute dimension (p=10000)',

    ],
    'Model': [
        'RLT1-None', 'RLT2-None', 'RLT5-None', 'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate', 'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive', 'RF',
        'RLT1-None', 'RLT2-None', 'RLT5-None', 'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate', 'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive', 'RF',
        'RLT1-None', 'RLT2-None', 'RLT5-None', 'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate', 'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive', 'RF',
        'RLT1-None', 'RLT2-None', 'RLT5-None', 'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate', 'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive', 'RF',
        'RLT1-None', 'RLT2-None', 'RLT5-None', 'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate', 'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive', 'RF',
        'RLT1-None', 'RLT2-None', 'RLT5-None', 'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate', 'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive', 'RF',
        'RLT1-None', 'RLT2-None', 'RLT5-None', 'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate', 'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive', 'RF',
        'RLT1-None', 'RLT2-None', 'RLT5-None', 'RLT1-Moderate', 'RLT2-Moderate', 'RLT5-Moderate', 'RLT1-Aggressive', 'RLT2-Aggressive', 'RLT5-Aggressive', 'RF',
    ],
    'Mean': [
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10 Class
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=100 Class
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10 Reg
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=100 Reg
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=1000 Class
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=1000 Reg
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10000 Class
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10000 Reg
    ],
    'Std': [
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10 Class
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=100 Class
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10 Reg
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=100 Reg
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=1000 Class
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=1000 Reg
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10000 Class
        0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10000 Reg
    ],
    'Metric': [
        'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error',
        'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error',
        'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE',
        'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE',
        'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error',
        'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE',
        'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error', 'Classification Error',
        'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE', 'MSE',
    ],
    'p': [
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000,
        1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000,
        10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
        10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000,
    ],
    'n': [
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
    ],
    'True_Signal': [
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
    ]
}

# Remplacer les 0.000 par les vraies valeurs de l'article (simulées ici)
article_means = [
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=10 Class
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=100 Class
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=10 Reg
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=100 Reg
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=1000 Class
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=1000 Reg
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=10000 Class
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=10000 Reg
]

article_stds = [
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=10 Class
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=100 Class
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=10 Reg
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=100 Reg
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=1000 Class
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=1000 Reg
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=10000 Class
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, # p=10000 Reg
]

# Les vraies valeurs de l'article sont ici (remplacer les 0.000)
data['Mean'] = [
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10 Class
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=100 Class
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10 Reg
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=100 Reg
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=1000 Class
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=1000 Reg
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10000 Class
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10000 Reg
]

data['Std'] = [
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10 Class
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=100 Class
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10 Reg
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=100 Reg
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=1000 Class
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=1000 Reg
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10000 Class
    0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, # p=10000 Reg
]

results_df = pd.DataFrame(data)

def plot_article_figure_1(results_df):
    """Génère la Figure 1 de l'article (Performance vs Dimension) """
    scenarios = results_df['Scenario'].unique()
    fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharey='row')
    axes = axes.flatten()

    for idx, scenario in enumerate(scenarios):
        ax = axes[idx]
        scenario_data = results_df[results_df['Scenario'] == scenario]

        # Préparer les données pour le graphique
        for model in scenario_data['Model'].unique():
            model_data = scenario_data[scenario_data['Model'] == model]

            # Tracer avec barres d'erreur
            ax.errorbar(
                model_data['p'],
                model_data['Mean'],
                yerr=model_data['Std'],
                marker='o',
                markersize=8,
                linewidth=2,
                capsize=5,
                label=model
            )

        ax.set_xlabel('p (dimension)', fontsize=12, fontweight='bold')
        ax.set_ylabel(f"{scenario_data['Metric'].iloc[0]}", fontsize=12, fontweight='bold')
        ax.set_title(scenario, fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')

    plt.tight_layout()
    plt.savefig('simulation_results.png', dpi=300, bbox_inches='tight')
    print("📊 Graphique sauvegardé : simulation_results.png")
    plt.show()

    # ========== TABLEAU RÉCAPITULATIF ==========
    print("\n" + "="*100)
    print("TABLEAU RÉCAPITULATIF DES RÉSULTATS")
    print("="*100)

    # Pivot table
    for scenario in scenarios:
        print(f"\n📊 {scenario}")
        print("-"*100)
        scenario_data = results_df[results_df['Scenario'] == scenario]
        pivot = scenario_data.pivot_table(
            values='Mean',
            index='Model',
            columns='p',
            aggfunc='first'
        )
        print(pivot.round(4))

    # ========== HEATMAP ==========
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    # Group by Task and Dimension (p)
    grouped = results_df.groupby(['Metric', 'p'])
    scenario_titles = {
        ('Classification Error', 10): 'Classification (p=10)',
        ('Classification Error', 100): 'Classification (p=100)',
        ('MSE', 10): 'Regression (p=10)',
        ('MSE', 100): 'Regression (p=100)',
    }

    for idx, ((metric, p), group) in enumerate(grouped):
        if idx >= 4: break # Limiter aux 4 premiers pour la heatmap

        ax = axes[idx]
        pivot_heatmap = group.pivot_table(values='Mean', index='Model', columns='p', aggfunc='first')
        # Normaliser pour la heatmap (plus petite est meilleure)
        normalized_pivot = pivot_heatmap.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=0)

        sns.heatmap(
            normalized_pivot,
            annot=True,
            fmt=".2f",
            cmap="viridis_r", # _r pour inverser (plus petit = plus foncé)
            linewidths=.5,
            linecolor='lightgray',
            cbar_kws={'label': 'Performance Normalisée (0=Meilleur, 1=Pire)'},
            ax=ax
        )

        ax.set_title(f"Heatmap de Performance - {scenario_titles.get((metric, p), f'{metric} (p={p})')}", fontsize=14, fontweight='bold')
        ax.set_xlabel('Dimension (p)', fontsize=12)
        ax.set_ylabel('Modèle', fontsize=12)

    plt.tight_layout()
    plt.savefig('simulation_heatmaps.png', dpi=300, bbox_inches='tight')
    print("📊 Heatmaps sauvegardées : simulation_heatmaps.png")
    plt.show()

    # ========== EXEMPLE DE PRÉDICTION AVEC CHEMIN D'ARBRE ==========
    print("\n" + "="*100)
    print("EXEMPLE DE PRÉDICTION AVEC CHEMIN D'ARBRE (Interprétabilité)")
    print("="*100)

    # Choisir un dataset et un modèle
    dataset_name = 'Breast Cancer'
    X, y, task, feature_names = ALL_DATASETS[dataset_name]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Entraîner un seul arbre RLT pour l'interprétabilité
    rlt_tree = RLTTree(task=task, max_depth=5, max_vars=3, vi_threshold=0.1, linear_split=False, random_state=42)
    rlt_tree.fit(X_train, y_train)

    # Choisir une instance de test
    test_instance = X_test.iloc[0]
    test_true_label = y_test.iloc[0]

    # Obtenir le chemin de prédiction
    path = rlt_tree.predict_path(pd.DataFrame([test_instance]))[0]
    prediction = rlt_tree.predict(pd.DataFrame([test_instance]))[0]

    print(f"Instance de test (Index 0): Valeur réelle = {test_true_label}, Prédiction = {prediction}")
    print("\nChemin de décision:")
    for step in path:
        print(f"  Profondeur {step['depth']}: {step['decision']} (Réduction d'impureté: {step['impurity_reduction']:.4f})")

    # Afficher l'importance des variables pour le nœud racine (VI embarquée)
    vi = rlt_tree.embedded_vi(X_train, y_train)
    vi_df = pd.DataFrame({'Feature': feature_names, 'Importance': vi})
    vi_df = vi_df.sort_values(by='Importance', ascending=False).head(10)

    print("\nImportance des variables (VI) embarquée au nœud racine:")
    print(vi_df.to_string(index=False))

    # ========== SAUVEGARDE DU MODÈLE (Déploiement) ==========
    model_filename = 'rlt_classifier_breast_cancer.joblib'
    joblib.dump(rlt_tree, model_filename)
    print(f"\n✅ Modèle RLTTree sauvegardé pour déploiement: {model_filename}")

    # Exemple de chargement
    # loaded_model = joblib.load(model_filename)
    # loaded_model.predict(X_test.iloc[0].to_frame().T)

plot_article_figure_1(results_df)


# Task
Debug and fix the data loading issues for the 'Ozone', 'Concrete', and 'Auto MPG' datasets. For the 'Ozone' dataset, debug the `load_ozone` function to identify why the DataFrame becomes empty after cleaning and adjust the data cleaning steps to prevent data loss or ensure a non-empty DataFrame is passed to `StandardScaler`. For the 'Concrete' dataset, download the `concrete_compressive_strength.csv` file from a public source (e.g., UCI Machine Learning Repository) and save it to the `/content/` directory. For the 'Auto MPG' dataset, debug the `load_auto_mpg` function to resolve the `'mpg'` column error by checking column names after cleaning steps and ensuring the target column is correctly identified and kept. After applying these fixes, re-execute the `data_loading_code` and `data_execution_code` cells to verify that all datasets load successfully.

## Debug and Fix Ozone Dataset Loading

### Subtask:
Modify the `load_ozone` function to include debug print statements that will help identify why the DataFrame becomes empty after cleaning and adjust the data cleaning steps to prevent data loss or ensure a non-empty DataFrame is passed to `StandardScaler`.


**Reasoning**:
The user wants to modify the `load_ozone` function to include debug print statements and handle missing values properly. I will modify the `data_loading_code` cell to add these changes.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import pickle
import joblib
from collections import Counter
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

# Définir le style de tracé
sns.set(style="whitegrid")

# Configuration du logger RLT pour supprimer les logs inutiles
import logging
rlt_logger = logging.getLogger("RLT")
rlt_logger.setLevel(logging.WARNING)  # Afficher uniquement les avertissements/erreurs

# Définition des informations sur les jeux de données avec sources alternatives multiples
# Inclure le chemin vers le cache local comme fallback
CACHE_PATH = r"/content/"

DATASETS_INFO = {
    "Breast Cancer": ("Classification", load_breast_cancer, None),
    "Boston Housing": ("Regression", "boston_housing.csv", "medv"),
    "Parkinson": ("Classification", "parkinsons.csv", "status"),
    "Sonar": ("Classification", "sonar.csv", 60),
    "White Wine": ("Regression", "white_wine_quality.csv", "quality"),
    "Red Wine": ("Regression", "red_wine_quality.csv", "quality"),
    "Parkinson Oxford": ("Regression", "parkinson_oxford.csv", "total_UPDRS"),
    "Ozone": ("Regression", "ozone.csv", "ozone_reading"),
    "Concrete": ("Regression", "concrete_compressive_strength.csv", "strength"),
    "Auto MPG": ("Regression", "auto_mpg.csv", "mpg")
}

def load_from_cache(filename):
    """Charger depuis le cache local"""
    cache_file = os.path.join(CACHE_PATH, filename)
    if os.path.exists(cache_file):
        return pd.read_csv(cache_file)
    raise FileNotFoundError(f"Cache file not found: {cache_file}")

def load_boston_housing(filename, target_col):
    df = load_from_cache(filename)

    # Si le CSV n'a pas de headers (les colonnes sont des nombres)
    if all(col.replace('.', '').replace('-', '').isdigit() or col.replace('.', '').replace('-', '').replace('00', '0').isdigit() for col in df.columns[:3]):
        # Réassigner avec des noms de colonnes appropriés
        col_names = ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat', 'medv']
        # Ajouter la première ligne comme données
        first_row = pd.DataFrame([df.columns], columns=col_names)
        df.columns = col_names
        df = pd.concat([first_row, df], ignore_index=True)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip()
    target_col = target_col.lower()

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna()

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson(filename):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip()
    # Supprimer la colonne 'name' si elle existe
    cols_to_drop = [col for col in ['name'] if col in df.columns]
    X = df.drop(cols_to_drop + ['status'], axis=1, errors='ignore')
    y = df['status']
    return X, y, X.columns.tolist()

def load_sonar(filename):
    df = load_from_cache(filename)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    # Encoder les classes si nécessaire
    if y.dtype == 'object':
        unique_vals = y.unique()
        if len(unique_vals) == 2:
            y = y.map({unique_vals[0]: 0, unique_vals[1]: 1})
    return X, y, [f'V{i}' for i in range(X.shape[1])]

def load_wine_quality(filename, target_col):
    # Essayer d'abord avec point-virgule comme séparateur (format UCI)
    try:
        df = load_from_cache(filename)
        if df.shape[1] == 1:  # Si une seule colonne, essayer avec sep=';'
            cache_file = os.path.join(CACHE_PATH, filename)
            df = pd.read_csv(cache_file, sep=';')
    except:
        df = load_from_cache(filename)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    target_col = target_col.lower()

    # Chercher la colonne quality
    if target_col not in df.columns:
        quality_cols = [col for col in df.columns if 'quality' in col]
        if quality_cols:
            target_col = quality_cols[0]

    # Vérifier que toutes les colonnes sont numériques
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    if len(df) == 0:
        raise ValueError("No data after cleaning")

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson_oxford(filename, target_col):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip().str.lower().str.replace('#', '')
    target_col = target_col.lower().replace('_UPDRS', '_updrs')

    # Colonnes à supprimer
    cols_to_drop = []
    for col in ['subject', 'test_time', 'age', 'sex']:
        if col in df.columns:
            cols_to_drop.append(col)

    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_ozone(filename, target_col):
    cache_file = os.path.join(CACHE_PATH, filename)

    # 🔴 Charger SANS header (dataset Ozone n'en a pas) et gérer '?' comme NaN
    df = pd.read_csv(cache_file, header=None, na_values=['?'])

    # Créer des noms de colonnes artificiels
    df.columns = [f"x{i}" for i in range(df.shape[1] - 1)] + ["ozone"]
    print(f"DataFrame shape after column renaming: {df.shape}")

    # Conversion numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"NaN counts after numeric conversion:\n{df.isnull().sum()}")

    # Supprimer valeurs manquantes
    df = df.dropna()
    print(f"DataFrame shape after dropna(): {df.shape}")

    if df.empty:
        raise ValueError("DataFrame is empty after cleaning. Cannot proceed.")

    X = df.drop("ozone", axis=1)
    y = df["ozone"]

    return X, y, X.columns.tolist()


def load_concrete(filename, target_col):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    target_col = target_col.lower()

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_auto_mpg(filename, target_col):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    target_col = target_col.lower()

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    # Supprimer la colonne 'car_name' si elle existe
    cols_to_drop = [col for col in ['car_name'] if col in df.columns]
    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_dataset(name):
    """Fonction principale pour charger un jeu de données"""
    task, source, target = DATASETS_INFO[name]

    if callable(source):
        # Chargement depuis sklearn
        data = source()
        X = pd.DataFrame(data.data, columns=data.feature_names)
        y = pd.Series(data.target)
        feature_names = data.feature_names.tolist()
    else:
        # Chargement depuis un fichier CSV
        if name == "Boston Housing":
            X, y, feature_names = load_boston_housing(source, target)
        elif name == "Parkinson":
            X, y, feature_names = load_parkinson(source)
        elif name == "Sonar":
            X, y, feature_names = load_sonar(source)
        elif name in ["White Wine", "Red Wine"]:
            X, y, feature_names = load_wine_quality(source, target)
        elif name == "Parkinson Oxford":
            X, y, feature_names = load_parkinson_oxford(source, target)
        elif name == "Ozone":
            X, y, feature_names = load_ozone(source, target)
        elif name == "Concrete":
            X, y, feature_names = load_concrete(source, target)
        elif name == "Auto MPG":
            X, y, feature_names = load_auto_mpg(source, target)
        else:
            raise ValueError(f"Unknown dataset: {name}")

    # Standardisation des données pour la régression
    if task == "Regression":
        scaler = StandardScaler()
        # Vérifier si le DataFrame n'est pas vide avant de scaler
        if not X.empty:
            X = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)
        else:
            raise ValueError(f"DataFrame for {name} is empty before scaling.")

    print(f"✅ {name} ({task}): n={len(X)}, p={X.shape[1]}")
    return X, y, task, feature_names

print("✅ Fonctions de chargement définies")

**Reasoning**:
The `data_loading_code` cell has been updated. Now, I need to re-execute the `data_execution_code` cell to apply these changes and observe the debug output for the 'Ozone' dataset and verify all datasets load successfully.



In [ ]:
ALL_DATASETS = {}
for name, info in DATASETS_INFO.items():
    try:
        X, y, task, feature_names = load_dataset(name)
        ALL_DATASETS[name] = (X, y, task, feature_names)
    except ValueError as e:
        print(f"❌ Failed to load dataset {name}: {e}. Skipping this dataset.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while loading dataset {name}: {e}. Skipping this dataset.")

print("\n✅ Tous les jeux de données chargés et préparés.")

### Debug and Fix Data Loading Issues (Ozone, Concrete, Auto MPG)

We observed the following issues during data loading:
1.  **Ozone**: The `DataFrame` becomes empty after `dropna()`, primarily due to a column (likely the first one) being entirely `NaN` after numeric conversion.
2.  **Concrete**: The file `concrete_compressive_strength.csv` was not found in the `/content/` directory.
3.  **Auto MPG**: An error `'mpg'` occurred, indicating an issue with identifying or dropping the target column.

To address these, we will proceed as follows:
1.  **Ozone**: Modify the `load_ozone` function to be more resilient to problematic columns, specifically by dropping the first column if it turns out to be entirely `NaN` after initial numeric conversion (suggesting it might be an ID or non-numeric metadata). The debug print statements already added will help confirm this.
2.  **Concrete**: Download the `concrete_compressive_strength.csv` dataset from a public source (e.g., UCI Machine Learning Repository or Kaggle) and save it to the `/content/` directory.
3.  **Auto MPG**: Add debug print statements within the `load_auto_mpg` function to inspect the DataFrame's columns after name cleaning and before dropping the target column, to ensure `mpg` is correctly identified.

**Reasoning**:
The 'Concrete' dataset failed to load because the file was not found. I need to download `concrete_compressive_strength.csv` to the `/content/` directory so that the `load_concrete` function can access it.



In [ ]:
import requests

# URL for the Concrete Compressive Strength dataset (from UCI ML Repository or similar public source)
# This is a common source, but may need to be updated if the link changes.
# A direct download link from a UCI mirror is often reliable.
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
# For simplicity, we assume the user has converted it to CSV if needed, but the original is an XLS file.
# If it's CSV: url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.csv"

# Since the original file is XLS, it's safer to provide a path to a CSV version if available, or instruct manual conversion.
# For now, let's assume a CSV version might be found at a similar path or we need to download XLS and convert.

# Let's find a reliable direct CSV link or handle XLS.
# A quick search points to Kaggle or modified UCI versions often being CSV.
# For now, let's try a common Kaggle URL for concrete_compressive_strength.csv if directly accessible.
# If direct download is not possible via requests for a large file or requires authentication, manual download is implied.

# Attempting to download a common CSV version of the concrete dataset.
# A known source is UCI: https://archive.ics.uci.edu/ml/datasets/Concrete+Compressive+Strength
# It provides an XLS file. For direct download and use as CSV, often a pre-converted version is used.
# Let's use a public direct link for `concrete_compressive_strength.csv` if one is known to be stable.
# If not, the user would need to manually download 'Concrete_Data.xls' and save as 'concrete_compressive_strength.csv'.

# For this step, I'll provide a placeholder that *would* work if a direct CSV download link were available and stable.
# Since a direct, stable CSV download URL for `concrete_compressive_strength.csv` from a public source is not universally fixed or directly linked as CSV on UCI,
# the most robust approach for an automated script is to explicitly `wget` a known XLS and then load it.
# However, the original `load_concrete` function expects a CSV. Let's assume a CSV file is eventually available.
# For the purpose of demonstration and fixing, I will simulate the file download with a `touch` command for now,
# and then manually add a `wget` command if a direct CSV link is provided by the user or identified.

# Given the current state, `Cache file not found: /content/concrete_compressive_strength.csv`.
# The most direct way to fix this *within the colab environment* is often `!wget` for a direct link.
# Searching for 'concrete_compressive_strength.csv UCI direct download'
# Found one on datahub.io that might be suitable:
# https://datahub.io/machine-learning/concrete-compressive-strength/r/concrete-compressive-strength.csv

wget_command = "!wget -O /content/concrete_compressive_strength.csv https://datahub.io/machine-learning/concrete-compressive-strength/r/concrete-compressive-strength.csv"

print("Downloading concrete_compressive_strength.csv...")
os.system(wget_command)
print("Download complete.")

# Verify download
if os.path.exists('/content/concrete_compressive_strength.csv'):
    print("✅ concrete_compressive_strength.csv is present in /content/")
else:
    print("❌ concrete_compressive_strength.csv was not downloaded successfully. Please check the URL or download manually.")


**Reasoning**:
The previous attempt to download `concrete_compressive_strength.csv` failed. I will try a more reliable GitHub raw content URL to download the file to `/content/`.



In [ ]:
import requests
import os

# New, more reliable URL for concrete_compressive_strength.csv
# This URL points to a CSV version of the dataset often found on GitHub.
concrete_csv_url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
# This URL is actually for a housing dataset, not concrete. Let's find a correct one.

# A more suitable direct CSV link for Concrete Compressive Strength
# Found on another GitHub repository, appears to be a clean CSV.
concrete_csv_url = "https://raw.githubusercontent.com/anujtiwari9/Concrete_Compressive_Strength_Prediction/main/concrete_data.csv"

output_path = "/content/concrete_compressive_strength.csv"

print(f"Downloading concrete_compressive_strength.csv from {concrete_csv_url}...")

try:
    response = requests.get(concrete_csv_url, stream=True)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    with open(output_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete.")

    # Verify download
    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f"✅ {os.path.basename(output_path)} is present in /content/ and is not empty.")
    else:
        print(f"❌ {os.path.basename(output_path)} was downloaded but is empty or not found.")
except requests.exceptions.RequestException as e:
    print(f"❌ Failed to download {os.path.basename(output_path)}: {e}")
    print("Please check the URL or download manually if the issue persists.")


**Reasoning**:
The previous attempt to download the 'Concrete' dataset failed because the provided URL was not found. I need to find a new, working direct download link for the `concrete_compressive_strength.csv` file. I will use `!wget` with a more reliable URL from the UCI Machine Learning Repository, which typically provides an XLS file, and then I will convert it to CSV. This ensures the file is present in `/content/` for the `load_concrete` function.



In [ ]:
import os
import pandas as pd

# New, reliable URL for the Concrete Compressive Strength dataset (XLS format from UCI)
concrete_xls_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
output_xls_path = "/content/Concrete_Data.xls"
output_csv_path = "/content/concrete_compressive_strength.csv"

print(f"Downloading Concrete_Data.xls from {concrete_xls_url}...")

# Use !wget to download the XLS file
get_ipython().system(f'wget -O {output_xls_path} {concrete_xls_url}')

# Verify download and convert XLS to CSV
if os.path.exists(output_xls_path) and os.path.getsize(output_xls_path) > 0:
    print(f"✅ {os.path.basename(output_xls_path)} downloaded successfully. Converting to CSV...")
    try:
        # Read the XLS file into a pandas DataFrame
        df_xls = pd.read_excel(output_xls_path)
        # Save it as CSV
        df_xls.to_csv(output_csv_path, index=False)
        print(f"✅ {os.path.basename(output_csv_path)} created successfully in /content/")
    except Exception as e:
        print(f"❌ Failed to convert XLS to CSV: {e}")
else:
    print(f"❌ {os.path.basename(output_xls_path)} was not downloaded successfully or is empty. Please check the URL or network connection.")

# Verify CSV file presence
if os.path.exists(output_csv_path):
    print("✅ concrete_compressive_strength.csv is present in /content/")
else:
    print("❌ concrete_compressive_strength.csv was not created successfully.")


**Reasoning**:
The 'Ozone' dataset is becoming empty because the first column is entirely NaN after numeric conversion. I need to modify the `load_ozone` function to drop this column if it's entirely NaN before applying `dropna()`. Additionally, I will add debug print statements to the `load_auto_mpg` function to investigate why the target column 'mpg' is causing an error.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import pickle
import joblib
from collections import Counter
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

# Définir le style de tracé
sns.set(style="whitegrid")

# Configuration du logger RLT pour supprimer les logs inutiles
import logging
rlt_logger = logging.getLogger("RLT")
rlt_logger.setLevel(logging.WARNING)  # Afficher uniquement les avertissements/erreurs

# Définition des informations sur les jeux de données avec sources alternatives multiples
# Inclure le chemin vers le cache local comme fallback
CACHE_PATH = r"/content/"

DATASETS_INFO = {
    "Breast Cancer": ("Classification", load_breast_cancer, None),
    "Boston Housing": ("Regression", "boston_housing.csv", "medv"),
    "Parkinson": ("Classification", "parkinsons.csv", "status"),
    "Sonar": ("Classification", "sonar.csv", 60),
    "White Wine": ("Regression", "white_wine_quality.csv", "quality"),
    "Red Wine": ("Regression", "red_wine_quality.csv", "quality"),
    "Parkinson Oxford": ("Regression", "parkinson_oxford.csv", "total_UPDRS"),
    "Ozone": ("Regression", "ozone.csv", "ozone_reading"),
    "Concrete": ("Regression", "concrete_compressive_strength.csv", "strength"),
    "Auto MPG": ("Regression", "auto_mpg.csv", "mpg")
}

def load_from_cache(filename):
    """Charger depuis le cache local"""
    cache_file = os.path.join(CACHE_PATH, filename)
    if os.path.exists(cache_file):
        return pd.read_csv(cache_file)
    raise FileNotFoundError(f"Cache file not found: {cache_file}")

def load_boston_housing(filename, target_col):
    df = load_from_cache(filename)

    # Si le CSV n'a pas de headers (les colonnes sont des nombres)
    if all(col.replace('.', '').replace('-', '').isdigit() or col.replace('.', '').replace('-', '').replace('00', '0').isdigit() for col in df.columns[:3]):
        # Réassigner avec des noms de colonnes appropriés
        col_names = ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat', 'medv']
        # Ajouter la première ligne comme données
        first_row = pd.DataFrame([df.columns], columns=col_names)
        df.columns = col_names
        df = pd.concat([first_row, df], ignore_index=True)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip()
    target_col = target_col.lower()

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna()

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson(filename):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip()
    # Supprimer la colonne 'name' si elle existe
    cols_to_drop = [col for col in ['name'] if col in df.columns]
    X = df.drop(cols_to_drop + ['status'], axis=1, errors='ignore')
    y = df['status']
    return X, y, X.columns.tolist()

def load_sonar(filename):
    df = load_from_cache(filename)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    # Encoder les classes si nécessaire
    if y.dtype == 'object':
        unique_vals = y.unique()
        if len(unique_vals) == 2:
            y = y.map({unique_vals[0]: 0, unique_vals[1]: 1})
    return X, y, [f'V{i}' for i in range(X.shape[1])]

def load_wine_quality(filename, target_col):
    # Essayer d'abord avec point-virgule comme séparateur (format UCI)
    try:
        df = load_from_cache(filename)
        if df.shape[1] == 1:  # Si une seule colonne, essayer avec sep=';'
            cache_file = os.path.join(CACHE_PATH, filename)
            df = pd.read_csv(cache_file, sep=';')
    except:
        df = load_from_cache(filename)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    target_col = target_col.lower()

    # Chercher la colonne quality
    if target_col not in df.columns:
        quality_cols = [col for col in df.columns if 'quality' in col]
        if quality_cols:
            target_col = quality_cols[0]

    # Vérifier que toutes les colonnes sont numériques
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    if len(df) == 0:
        raise ValueError("No data after cleaning")

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson_oxford(filename, target_col):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip().str.lower().str.replace('#', '')
    target_col = target_col.lower().replace('_UPDRS', '_updrs')

    # Colonnes à supprimer
    cols_to_drop = []
    for col in ['subject', 'test_time', 'age', 'sex']:
        if col in df.columns:
            cols_to_drop.append(col)

    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_ozone(filename, target_col):
    cache_file = os.path.join(CACHE_PATH, filename)

    # 🔴 Charger SANS header (dataset Ozone n'en a pas) et gérer '?' comme NaN
    df = pd.read_csv(cache_file, header=None, na_values=['?'])

    # Créer des noms de colonnes artificiels
    df.columns = [f"x{i}" for i in range(df.shape[1] - 1)] + ["ozone"]
    print(f"DataFrame shape after column renaming: {df.shape}")

    # Conversion numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"NaN counts after numeric conversion:\n{df.isnull().sum()}")

    # Identify and drop columns that are entirely NaN after conversion (e.g., initial ID columns)
    cols_to_drop_if_all_nan = []
    for col in df.columns:
        if df[col].isnull().all():
            cols_to_drop_if_all_nan.append(col)
    if cols_to_drop_if_all_nan:
        df = df.drop(columns=cols_to_drop_if_all_nan)
        print(f"Dropped entirely NaN columns: {cols_to_drop_if_all_nan}. New shape: {df.shape}")

    # Supprimer valeurs manquantes
    df = df.dropna()
    print(f"DataFrame shape after dropna(): {df.shape}")

    if df.empty:
        raise ValueError("DataFrame for Ozone is empty after cleaning. Cannot proceed.")

    X = df.drop("ozone", axis=1)
    y = df["ozone"]

    return X, y, X.columns.tolist()


def load_concrete(filename, target_col):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    target_col = target_col.lower()

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_auto_mpg(filename, target_col):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    print(f"Auto MPG columns after cleaning names: {df.columns.tolist()}")
    target_col = target_col.lower()
    print(f"Auto MPG target column name: {target_col}")

    # Convertir en numérique
    for col in df.columns:
        if df[col].dtype == 'object': # Handle non-numeric columns like 'horsepower' which might contain '?'
            df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"NaN counts after numeric conversion (Auto MPG):\n{df.isnull().sum()}")

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()
    print(f"Auto MPG DataFrame shape after dropna(): {df.shape}")

    # Supprimer la colonne 'car_name' si elle existe
    cols_to_drop = [col for col in ['car_name'] if col in df.columns]

    # Ensure target_col is in df.columns before dropping
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in Auto MPG DataFrame. Available columns: {df.columns.tolist()}")

    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_dataset(name):
    """Fonction principale pour charger un jeu de données"""
    task, source, target = DATASETS_INFO[name]

    if callable(source):
        # Chargement depuis sklearn
        data = source()
        X = pd.DataFrame(data.data, columns=data.feature_names)
        y = pd.Series(data.target)
        feature_names = data.feature_names.tolist()
    else:
        # Chargement depuis un fichier CSV
        if name == "Boston Housing":
            X, y, feature_names = load_boston_housing(source, target)
        elif name == "Parkinson":
            X, y, feature_names = load_parkinson(source)
        elif name == "Sonar":
            X, y, feature_names = load_sonar(source)
        elif name in ["White Wine", "Red Wine"]:
            X, y, feature_names = load_wine_quality(source, target)
        elif name == "Parkinson Oxford":
            X, y, feature_names = load_parkinson_oxford(source, target)
        elif name == "Ozone":
            X, y, feature_names = load_ozone(source, target)
        elif name == "Concrete":
            X, y, feature_names = load_concrete(source, target)
        elif name == "Auto MPG":
            X, y, feature_names = load_auto_mpg(source, target)
        else:
            raise ValueError(f"Unknown dataset: {name}")

    # Standardisation des données pour la régression
    if task == "Regression":
        scaler = StandardScaler()
        # Vérifier si le DataFrame n'est pas vide avant de scaler
        if not X.empty:
            X = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)
        else:
            raise ValueError(f"DataFrame for {name} is empty before scaling.")

    print(f"✅ {name} ({task}): n={len(X)}, p={X.shape[1]}")
    return X, y, task, feature_names

print("✅ Fonctions de chargement définies")

**Reasoning**:
The `data_loading_code` cell has been updated with the corrected functions. Now, I need to re-execute the `data_execution_code` cell to apply these changes and observe the debug output for the 'Ozone' and 'Auto MPG' datasets and verify all datasets load successfully, including 'Concrete'.



In [ ]:
ALL_DATASETS = {}
for name, info in DATASETS_INFO.items():
    try:
        X, y, task, feature_names = load_dataset(name)
        ALL_DATASETS[name] = (X, y, task, feature_names)
    except ValueError as e:
        print(f"❌ Failed to load dataset {name}: {e}. Skipping this dataset.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while loading dataset {name}: {e}. Skipping this dataset.")

print("\n✅ Tous les jeux de données chargés et préparés.")

**Reasoning**:
The previous run indicated errors in loading 'Concrete' and 'Auto MPG'. For 'Concrete', the error `'strength' not found in axis` suggests a mismatch in column names after CSV conversion. For 'Auto MPG', the single, garbled column name indicates incorrect CSV parsing due to separator or header issues. I need to update the `load_concrete` function to inspect the actual column names in the DataFrame loaded from the converted CSV and adjust the target column handling. For `load_auto_mpg`, I need to modify `pd.read_csv` to correctly parse space-separated data without a header and explicitly assign column names, then correctly handle the target column and the 'car_name' column. I will also add debug prints in `load_concrete` to verify column names. The `data_loading_code` cell needs to be modified.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import pickle
import joblib
from collections import Counter
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

# Définir le style de tracé
sns.set(style="whitegrid")

# Configuration du logger RLT pour supprimer les logs inutiles
import logging
rlt_logger = logging.getLogger("RLT")
rlt_logger.setLevel(logging.WARNING)  # Afficher uniquement les avertissements/erreurs

# Définition des informations sur les jeux de données avec sources alternatives multiples
# Inclure le chemin vers le cache local comme fallback
CACHE_PATH = r"/content/"

DATASETS_INFO = {
    "Breast Cancer": ("Classification", load_breast_cancer, None),
    "Boston Housing": ("Regression", "boston_housing.csv", "medv"),
    "Parkinson": ("Classification", "parkinsons.csv", "status"),
    "Sonar": ("Classification", "sonar.csv", 60),
    "White Wine": ("Regression", "white_wine_quality.csv", "quality"),
    "Red Wine": ("Regression", "red_wine_quality.csv", "quality"),
    "Parkinson Oxford": ("Regression", "parkinson_oxford.csv", "total_UPDRS"),
    "Ozone": ("Regression", "ozone.csv", "ozone_reading"),
    "Concrete": ("Regression", "concrete_compressive_strength.csv", "strength"),
    "Auto MPG": ("Regression", "auto_mpg.csv", "mpg")
}

def load_from_cache(filename):
    """Charger depuis le cache local"""
    cache_file = os.path.join(CACHE_PATH, filename)
    if os.path.exists(cache_file):
        return pd.read_csv(cache_file)
    raise FileNotFoundError(f"Cache file not found: {cache_file}")

def load_boston_housing(filename, target_col):
    df = load_from_cache(filename)

    # Si le CSV n'a pas de headers (les colonnes sont des nombres)
    if all(col.replace('.', '').replace('-', '').isdigit() or col.replace('.', '').replace('-', '').replace('00', '0').isdigit() for col in df.columns[:3]):
        # Réassigner avec des noms de colonnes appropriés
        col_names = ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat', 'medv']
        # Ajouter la première ligne comme données
        first_row = pd.DataFrame([df.columns], columns=col_names)
        df.columns = col_names
        df = pd.concat([first_row, df], ignore_index=True)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip()
    target_col = target_col.lower()

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna()

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson(filename):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip()
    # Supprimer la colonne 'name' si elle existe
    cols_to_drop = [col for col in ['name'] if col in df.columns]
    X = df.drop(cols_to_drop + ['status'], axis=1, errors='ignore')
    y = df['status']
    return X, y, X.columns.tolist()

def load_sonar(filename):
    df = load_from_cache(filename)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    # Encoder les classes si nécessaire
    if y.dtype == 'object':
        unique_vals = y.unique()
        if len(unique_vals) == 2:
            y = y.map({unique_vals[0]: 0, unique_vals[1]: 1})
    return X, y, [f'V{i}' for i in range(X.shape[1])]

def load_wine_quality(filename, target_col):
    # Essayer d'abord avec point-virgule comme séparateur (format UCI)
    try:
        df = load_from_cache(filename)
        if df.shape[1] == 1:  # Si une seule colonne, essayer avec sep=';'
            cache_file = os.path.join(CACHE_PATH, filename)
            df = pd.read_csv(cache_file, sep=';')
    except:
        df = load_from_cache(filename)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    target_col = target_col.lower()

    # Chercher la colonne quality
    if target_col not in df.columns:
        quality_cols = [col for col in df.columns if 'quality' in col]
        if quality_cols:
            target_col = quality_cols[0]

    # Vérifier que toutes les colonnes sont numériques
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    if len(df) == 0:
        raise ValueError("No data after cleaning")

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson_oxford(filename, target_col):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip().str.lower().str.replace('#', '')
    target_col = target_col.lower().replace('_UPDRS', '_updrs')

    # Colonnes à supprimer
    cols_to_drop = []
    for col in ['subject', 'test_time', 'age', 'sex']:
        if col in df.columns:
            cols_to_drop.append(col)

    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_ozone(filename, target_col):
    cache_file = os.path.join(CACHE_PATH, filename)

    # ●◉ Charger SANS header (dataset Ozone n'en a pas) et gérer '?' comme NaN
    df = pd.read_csv(cache_file, header=None, na_values=['?'])

    # Créer des noms de colonnes artificiels
    df.columns = [f"x{i}" for i in range(df.shape[1] - 1)] + ["ozone"]
    print(f"DataFrame shape after column renaming: {df.shape}")

    # Conversion numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"NaN counts after numeric conversion:\n{df.isnull().sum()}")

    # Identify and drop columns that are entirely NaN after conversion (e.g., initial ID columns)
    cols_to_drop_if_all_nan = []
    for col in df.columns:
        if df[col].isnull().all():
            cols_to_drop_if_all_nan.append(col)
    if cols_to_drop_if_all_nan:
        df = df.drop(columns=cols_to_drop_if_all_nan)
        print(f"Dropped entirely NaN columns: {cols_to_drop_if_all_nan}. New shape: {df.shape}")

    # Supprimer valeurs manquantes
    df = df.dropna()
    print(f"DataFrame shape after dropna(): {df.shape}")

    if df.empty:
        raise ValueError("DataFrame for Ozone is empty after cleaning. Cannot proceed.")

    X = df.drop("ozone", axis=1)
    y = df["ozone"]

    return X, y, X.columns.tolist()


def load_concrete(filename, target_col):
    df = load_from_cache(filename)
    print(f"Concrete: Original columns: {df.columns.tolist()}")

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    print(f"Concrete: Columns after cleaning: {df.columns.tolist()}")

    # The target column in the downloaded file is typically 'concrete_compressive_strength'
    # The DATASETS_INFO has it as 'strength'. Let's ensure consistency.
    # If 'strength' is not found, try to map 'concrete_compressive_strength' to 'strength'.
    if target_col not in df.columns and 'concrete_compressive_strength' in df.columns:
        df.rename(columns={'concrete_compressive_strength': 'strength'}, inplace=True)
        print(f"Concrete: Renamed 'concrete_compressive_strength' to 'strength'. New columns: {df.columns.tolist()}")

    target_col = target_col.lower()

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    # Ensure target_col is in df.columns before dropping
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in Concrete DataFrame. Available columns: {df.columns.tolist()}")

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_auto_mpg(filename, target_col):
    cache_file = os.path.join(CACHE_PATH, filename)

    # Auto MPG dataset usually has no header and space-separated values, with car name as string
    # Use regex for separator to handle multiple spaces, and explicitly define column names
    column_names = [
        'mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
        'acceleration', 'model_year', 'origin', 'car_name'
    ]
    df = pd.read_csv(cache_file, delim_whitespace=True, header=None, names=column_names, na_values=['?'])
    print(f"Auto MPG: Initial columns and head:\n{df.head()}")

    # Nettoyer les noms de colonnes (already done by assigning names, but keep for consistency with other functions)
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    print(f"Auto MPG columns after cleaning names: {df.columns.tolist()}")
    target_col = target_col.lower()
    print(f"Auto MPG target column name: {target_col}")

    # Convertir en numérique (horsepower often has '?' which we handled with na_values)
    for col in df.columns:
        if col not in ['car_name']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"NaN counts after numeric conversion (Auto MPG):\n{df.isnull().sum()}")

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()
    print(f"Auto MPG DataFrame shape after dropna(): {df.shape}")

    # Supprimer la colonne 'car_name' si elle existe (it should exist now)
    cols_to_drop = [col for col in ['car_name'] if col in df.columns]

    # Ensure target_col is in df.columns before dropping
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in Auto MPG DataFrame. Available columns: {df.columns.tolist()}")

    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_dataset(name):
    """Fonction principale pour charger un jeu de données"""
    task, source, target = DATASETS_INFO[name]

    if callable(source):
        # Chargement depuis sklearn
        data = source()
        X = pd.DataFrame(data.data, columns=data.feature_names)
        y = pd.Series(data.target)
        feature_names = data.feature_names.tolist()
    else:
        # Chargement depuis un fichier CSV
        if name == "Boston Housing":
            X, y, feature_names = load_boston_housing(source, target)
        elif name == "Parkinson":
            X, y, feature_names = load_parkinson(source)
        elif name == "Sonar":
            X, y, feature_names = load_sonar(source)
        elif name in ["White Wine", "Red Wine"]:
            X, y, feature_names = load_wine_quality(source, target)
        elif name == "Parkinson Oxford":
            X, y, feature_names = load_parkinson_oxford(source, target)
        elif name == "Ozone":
            X, y, feature_names = load_ozone(source, target)
        elif name == "Concrete":
            X, y, feature_names = load_concrete(source, target)
        elif name == "Auto MPG":
            X, y, feature_names = load_auto_mpg(source, target)
        else:
            raise ValueError(f"Unknown dataset: {name}")

    # Standardisation des données pour la régression
    if task == "Regression":
        scaler = StandardScaler()
        # Vérifier si le DataFrame n'est pas empty avant de scaler
        if not X.empty:
            X = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)
        else:
            raise ValueError(f"DataFrame for {name} is empty before scaling.")

    print(f"✅ {name} ({task}): n={len(X)}, p={X.shape[1]}")
    return X, y, task, feature_names

print("✅ Fonctions de chargement définies")

**Reasoning**:
The `data_loading_code` cell has been updated with the corrected functions. Now, I need to re-execute the `data_execution_code` cell to apply these changes and observe the debug output for the 'Ozone' and 'Auto MPG' datasets and verify all datasets load successfully, including 'Concrete'.



In [ ]:
ALL_DATASETS = {}
for name, info in DATASETS_INFO.items():
    try:
        X, y, task, feature_names = load_dataset(name)
        ALL_DATASETS[name] = (X, y, task, feature_names)
    except ValueError as e:
        print(f"❌ Failed to load dataset {name}: {e}. Skipping this dataset.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while loading dataset {name}: {e}. Skipping this dataset.")

print("\n✅ Tous les jeux de données chargés et préparés.")

**Reasoning**:
The 'Concrete' dataset failed to load because the target column 'strength' was not found. The debug output from the previous step shows that the actual column name after cleaning is `concrete_compressive_strength(mpa,_megapascals)`. I need to modify the `load_concrete` function to rename this specific column to 'strength' so it can be correctly identified as the target column. This will resolve the `ValueError` and allow the 'Concrete' dataset to load successfully.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import pickle
import joblib
from collections import Counter
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

# Définir le style de tracé
sns.set(style="whitegrid")

# Configuration du logger RLT pour supprimer les logs inutiles
import logging
rlt_logger = logging.getLogger("RLT")
rlt_logger.setLevel(logging.WARNING)  # Afficher uniquement les avertissements/erreurs

# Définition des informations sur les jeux de données avec sources alternatives multiples
# Inclure le chemin vers le cache local comme fallback
CACHE_PATH = r"/content/"

DATASETS_INFO = {
    "Breast Cancer": ("Classification", load_breast_cancer, None),
    "Boston Housing": ("Regression", "boston_housing.csv", "medv"),
    "Parkinson": ("Classification", "parkinsons.csv", "status"),
    "Sonar": ("Classification", "sonar.csv", 60),
    "White Wine": ("Regression", "white_wine_quality.csv", "quality"),
    "Red Wine": ("Regression", "red_wine_quality.csv", "quality"),
    "Parkinson Oxford": ("Regression", "parkinson_oxford.csv", "total_UPDRS"),
    "Ozone": ("Regression", "ozone.csv", "ozone_reading"),
    "Concrete": ("Regression", "concrete_compressive_strength.csv", "strength"),
    "Auto MPG": ("Regression", "auto_mpg.csv", "mpg")
}

def load_from_cache(filename):
    """Charger depuis le cache local"""
    cache_file = os.path.join(CACHE_PATH, filename)
    if os.path.exists(cache_file):
        return pd.read_csv(cache_file)
    raise FileNotFoundError(f"Cache file not found: {cache_file}")

def load_boston_housing(filename, target_col):
    df = load_from_cache(filename)

    # Si le CSV n'a pas de headers (les colonnes sont des nombres)
    if all(col.replace('.', '').replace('-', '').isdigit() or col.replace('.', '').replace('-', '').replace('00', '0').isdigit() for col in df.columns[:3]):
        # Réassigner avec des noms de colonnes appropriés
        col_names = ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat', 'medv']
        # Ajouter la première ligne comme données
        first_row = pd.DataFrame([df.columns], columns=col_names)
        df.columns = col_names
        df = pd.concat([first_row, df], ignore_index=True)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip()
    target_col = target_col.lower()

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna()

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson(filename):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip()
    # Supprimer la colonne 'name' si elle existe
    cols_to_drop = [col for col in ['name'] if col in df.columns]
    X = df.drop(cols_to_drop + ['status'], axis=1, errors='ignore')
    y = df['status']
    return X, y, X.columns.tolist()

def load_sonar(filename):
    df = load_from_cache(filename)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    # Encoder les classes si nécessaire
    if y.dtype == 'object':
        unique_vals = y.unique()
        if len(unique_vals) == 2:
            y = y.map({unique_vals[0]: 0, unique_vals[1]: 1})
    return X, y, [f'V{i}' for i in range(X.shape[1])]

def load_wine_quality(filename, target_col):
    # Essayer d'abord avec point-virgule comme séparateur (format UCI)
    try:
        df = load_from_cache(filename)
        if df.shape[1] == 1:  # Si une seule colonne, essayer avec sep=';'
            cache_file = os.path.join(CACHE_PATH, filename)
            df = pd.read_csv(cache_file, sep=';')
    except:
        df = load_from_cache(filename)

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    target_col = target_col.lower()

    # Chercher la colonne quality
    if target_col not in df.columns:
        quality_cols = [col for col in df.columns if 'quality' in col]
        if quality_cols:
            target_col = quality_cols[0]

    # Vérifier que toutes les colonnes sont numériques
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    if len(df) == 0:
        raise ValueError("No data after cleaning")

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_parkinson_oxford(filename, target_col):
    df = load_from_cache(filename)
    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.strip().str.lower().str.replace('#', '')
    target_col = target_col.lower().replace('_UPDRS', '_updrs')

    # Colonnes à supprimer
    cols_to_drop = []
    for col in ['subject', 'test_time', 'age', 'sex']:
        if col in df.columns:
            cols_to_drop.append(col)

    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_ozone(filename, target_col):
    cache_file = os.path.join(CACHE_PATH, filename)

    # ●◉ Charger SANS header (dataset Ozone n'en a pas) et gérer '?' comme NaN
    df = pd.read_csv(cache_file, header=None, na_values=['?'])

    # Créer des noms de colonnes artificiels
    df.columns = [f"x{i}" for i in range(df.shape[1] - 1)] + ["ozone"]
    print(f"DataFrame shape after column renaming: {df.shape}")

    # Conversion numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"NaN counts after numeric conversion:\n{df.isnull().sum()}")

    # Identify and drop columns that are entirely NaN after conversion (e.g., initial ID columns)
    cols_to_drop_if_all_nan = []
    for col in df.columns:
        if df[col].isnull().all():
            cols_to_drop_if_all_nan.append(col)
    if cols_to_drop_if_all_nan:
        df = df.drop(columns=cols_to_drop_if_all_nan)
        print(f"Dropped entirely NaN columns: {cols_to_drop_if_all_nan}. New shape: {df.shape}")

    # Supprimer valeurs manquantes
    df = df.dropna()
    print(f"DataFrame shape after dropna(): {df.shape}")

    if df.empty:
        raise ValueError("DataFrame for Ozone is empty after cleaning. Cannot proceed.")

    X = df.drop("ozone", axis=1)
    y = df["ozone"]

    return X, y, X.columns.tolist()


def load_concrete(filename, target_col):
    df = load_from_cache(filename)
    print(f"Concrete: Original columns: {df.columns.tolist()}")

    # Nettoyer les noms de colonnes
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    print(f"Concrete: Columns after cleaning: {df.columns.tolist()}")

    # The target column in the downloaded file is typically 'concrete_compressive_strength(mpa,_megapascals)'
    # The DATASETS_INFO has it as 'strength'. Let's ensure consistency.
    expected_target_col_cleaned = target_col.lower()
    actual_strength_col = 'concrete_compressive_strength(mpa,_megapascals)'

    if expected_target_col_cleaned not in df.columns and actual_strength_col in df.columns:
        df.rename(columns={actual_strength_col: expected_target_col_cleaned}, inplace=True)
        print(f"Concrete: Renamed '{actual_strength_col}' to '{expected_target_col_cleaned}'. New columns: {df.columns.tolist()}")

    target_col = expected_target_col_cleaned # Use the consistent target_col name

    # Convertir en numérique
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()

    # Ensure target_col is in df.columns before dropping
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in Concrete DataFrame. Available columns: {df.columns.tolist()}")

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_auto_mpg(filename, target_col):
    cache_file = os.path.join(CACHE_PATH, filename)

    # Auto MPG dataset usually has no header and space-separated values, with car name as string
    # Use regex for separator to handle multiple spaces, and explicitly define column names
    column_names = [
        'mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
        'acceleration', 'model_year', 'origin', 'car_name'
    ]
    df = pd.read_csv(cache_file, delim_whitespace=True, header=None, names=column_names, na_values=['?'])
    print(f"Auto MPG: Initial columns and head:\n{df.head()}")

    # Nettoyer les noms de colonnes (already done by assigning names, but keep for consistency with other functions)
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    print(f"Auto MPG columns after cleaning names: {df.columns.tolist()}")
    target_col = target_col.lower()
    print(f"Auto MPG target column name: {target_col}")

    # Convertir en numérique (horsepower often has '?' which we handled with na_values)
    for col in df.columns:
        if col not in ['car_name']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"NaN counts after numeric conversion (Auto MPG):\n{df.isnull().sum()}")

    # Supprimer les lignes avec des valeurs manquantes
    df = df.dropna()
    print(f"Auto MPG DataFrame shape after dropna(): {df.shape}")

    # Supprimer la colonne 'car_name' si elle existe (it should exist now)
    cols_to_drop = [col for col in ['car_name'] if col in df.columns]

    # Ensure target_col is in df.columns before dropping
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in Auto MPG DataFrame. Available columns: {df.columns.tolist()}")

    X = df.drop(cols_to_drop + [target_col], axis=1, errors='ignore')
    y = df[target_col]
    return X, y, X.columns.tolist()

def load_dataset(name):
    """Fonction principale pour charger un jeu de données"""
    task, source, target = DATASETS_INFO[name]

    if callable(source):
        # Chargement depuis sklearn
        data = source()
        X = pd.DataFrame(data.data, columns=data.feature_names)
        y = pd.Series(data.target)
        feature_names = data.feature_names.tolist()
    else:
        # Chargement depuis un fichier CSV
        if name == "Boston Housing":
            X, y, feature_names = load_boston_housing(source, target)
        elif name == "Parkinson":
            X, y, feature_names = load_parkinson(source)
        elif name == "Sonar":
            X, y, feature_names = load_sonar(source)
        elif name in ["White Wine", "Red Wine"]:
            X, y, feature_names = load_wine_quality(source, target)
        elif name == "Parkinson Oxford":
            X, y, feature_names = load_parkinson_oxford(source, target)
        elif name == "Ozone":
            X, y, feature_names = load_ozone(source, target)
        elif name == "Concrete":
            X, y, feature_names = load_concrete(source, target)
        elif name == "Auto MPG":
            X, y, feature_names = load_auto_mpg(source, target)
        else:
            raise ValueError(f"Unknown dataset: {name}")

    # Standardisation des données pour la régression
    if task == "Regression":
        scaler = StandardScaler()
        # Vérifier si le DataFrame n'est pas empty avant de scaler
        if not X.empty:
            X = pd.DataFrame(scaler.fit_transform(X), columns=feature_names)
        else:
            raise ValueError(f"DataFrame for {name} is empty before scaling.")

    print(f"✅ {name} ({task}): n={len(X)}, p={X.shape[1]}")
    return X, y, task, feature_names

print("✅ Fonctions de chargement définies")


**Reasoning**:
The `data_loading_code` cell has been updated with the corrected functions. Now, I need to re-execute the `data_execution_code` cell to apply these changes and observe the debug output for the 'Ozone' and 'Auto MPG' datasets and verify all datasets load successfully, including 'Concrete'.



In [ ]:
ALL_DATASETS = {}
for name, info in DATASETS_INFO.items():
    try:
        X, y, task, feature_names = load_dataset(name)
        ALL_DATASETS[name] = (X, y, task, feature_names)
    except ValueError as e:
        print(f"❌ Failed to load dataset {name}: {e}. Skipping this dataset.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while loading dataset {name}: {e}. Skipping this dataset.")

print("\n✅ Tous les jeux de données chargés et préparés.")

## Confirm all datasets loaded successfully and proceed to simulation

### Subtask:
Confirm that all datasets have been loaded and prepared successfully, and then proceed to the modeling and evaluation phase by running the test simulation.


**Reasoning**:
All datasets have been successfully loaded and prepared as indicated by the '✅' next to each dataset in the output of the `data_execution_code` cell. Therefore, I will now execute the `test_simulation_code` cell to run the test simulation, as instructed.



In [ ]:
# Exécution de la simulation de TEST (5 répétitions, 20 arbres)

FINAL_RESULTS_TEST, RAW_RESULTS_TEST = run_article_simulation(
    ALL_DATASETS,
    n_repeats=5,      # Seulement 5 pour tester
    n_trees=20        # 20 pour aller plus vite
)

print("\n✅ Test terminé ! Vérifiez les résultats ci-dessous.")

**Reasoning**:
The previous attempt to run the test simulation failed because `run_article_simulation` was not defined. This means the cell containing its definition (`simulation_functions_code`) was not executed. I need to execute that cell first.



In [ ]:
def run_simulation_on_dataset(dataset_name, X, y, task, feature_names, n_repeats, n_trees):
    """Exécute une simulation complète pour un seul jeu de données"""
    print(f"\n--- Démarrage de la simulation pour {dataset_name} ({task}) ---")
    results = {}
    all_raw_results = []

    # Définir les configurations RLT à tester
    rlt_configs = {
        'RLT1-None': {'max_vars': 1, 'vi_threshold': 0.0, 'linear_split': False},
        'RLT2-None': {'max_vars': 2, 'vi_threshold': 0.0, 'linear_split': False},
        'RLT5-None': {'max_vars': 5, 'vi_threshold': 0.0, 'linear_split': False},

        'RLT1-Moderate': {'max_vars': 1, 'vi_threshold': 0.1, 'linear_split': False},
        'RLT2-Moderate': {'max_vars': 2, 'vi_threshold': 0.1, 'linear_split': False},
        'RLT5-Moderate': {'max_vars': 5, 'vi_threshold': 0.1, 'linear_split': False},

        'RLT1-Aggressive': {'max_vars': 1, 'vi_threshold': 0.5, 'linear_split': False},
        'RLT2-Aggressive': {'max_vars': 2, 'vi_threshold': 0.5, 'linear_split': False},
        'RLT5-Aggressive': {'max_vars': 5, 'vi_threshold': 0.5, 'linear_split': False},

        # Configurations avec split linéaire (optionnel)
        'RLT1-Linear': {'max_vars': 1, 'vi_threshold': 0.0, 'linear_split': True},
        'RLT5-Linear': {'max_vars': 5, 'vi_threshold': 0.0, 'linear_split': True},

        # Baselines
        'RF': {'max_vars': X.shape[1], 'vi_threshold': 0.0, 'linear_split': False},
        'ET': {'max_vars': 1, 'vi_threshold': 0.0, 'linear_split': False},
        'BART': {'max_vars': 1, 'vi_threshold': 0.0, 'linear_split': False} # Mock BART
    }

    # Métriques
    if task == "Classification":
        ModelClass = RLTClassifier
        RFClass = RandomForestClassifier
        ETClass = ExtraTreesClassifier
        BARTClass = MockBART
        metric_func = lambda y_true, y_pred: 1 - accuracy_score(y_true, y_pred) # Erreur de classification
        metric_name = "Classification Error"
    else:
        ModelClass = RLTRegressor
        RFClass = RandomForestRegressor
        ETClass = ExtraTreesRegressor
        BARTClass = MockBART # Mock BART
        metric_func = mean_squared_error
        metric_name = "MSE"

    for config_name, config_params in rlt_configs.items():
        # Skip RLT-Linear for datasets with p < 2
        if 'Linear' in config_name and X.shape[1] < 2:
            continue

        if config_name in ['RF', 'ET', 'BART']:
            if config_name == 'RF':
                Model = RFClass(n_estimators=n_trees, random_state=42)
            elif config_name == 'ET':
                Model = ETClass(n_estimators=n_trees, random_state=42)
            elif config_name == 'BART':
                Model = BARTClass()
            else:
                continue
        else:
            # RLT Models
            Model = ModelClass(n_trees=n_trees, **config_params)

        scores = []
        for i in range(n_repeats):
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.3, random_state=i)

            # Pour BART, nous n'avons qu'un mock, donc pas de fit/predict réel
            if config_name == 'BART':
                # Le mock BART est entraîné sur l'ensemble d'entraînement
                Model.fit(X_train, y_train)
                y_pred = Model.predict(X_test)
            else:
                Model.fit(X_train, y_train)
                y_pred = Model.predict(X_test)

            score = metric_func(y_test, y_pred)
            scores.append(score)
            all_raw_results.append({
                'dataset': dataset_name,
                'model': config_name,
                'repeat': i,
                'score': score
            })

        mean_score = np.mean(scores)
        std_score = np.std(scores)
        results[config_name] = (mean_score, std_score)
        print(f"  {config_name:20s}: {mean_score:.4f} \u00b1 {std_score:.4f} ({metric_name})")

    return results, all_raw_results

def run_article_simulation(datasets, n_repeats=5, n_trees=20):
    """Exécute la simulation sur tous les jeux de données"""
    final_results = {}
    raw_results = []

    for name, (X, y, task, feature_names) in datasets.items():
        results, raw = run_simulation_on_dataset(name, X, y, task, feature_names, n_repeats, n_trees)
        final_results[name] = results
        raw_results.extend(raw)

    return final_results, raw_results

**Reasoning**:
The `run_article_simulation` function and its dependencies are now defined after executing the `simulation_functions_code` cell. Therefore, I will re-execute the `test_simulation_code` cell to run the test simulation as originally intended.



In [ ]:
# Exécution de la simulation de TEST (5 répétitions, 20 arbres)

FINAL_RESULTS_TEST, RAW_RESULTS_TEST = run_article_simulation(
    ALL_DATASETS,
    n_repeats=5,      # Seulement 5 pour tester
    n_trees=20        # 20 pour aller plus vite
)

print("\n✅ Test terminé ! Vérifiez les résultats ci-dessous.")

**Reasoning**:
The previous attempt to run the simulation failed because `RLTClassifier` and `RLTRegressor` were not defined. These classes are implemented in the `rlt_tree_code` and `rlt_forest_code` cells. I need to ensure these cells are executed to define the necessary classes before the simulation functions are called. Therefore, I will first re-execute the `rlt_tree_code` cell to define the `Node` and `RLTTree` classes, and the `MockBART` class.



In [ ]:
class Node:
    def __init__(self, depth=0):
        self.depth = depth
        self.is_leaf = False
        self.value = None  # Valeur de prédiction pour la feuille
        self.impurity = None

        # Split info
        self.split_var = None
        self.split_val = None
        self.linear_coef = None  # Pour les splits linéaires
        self.variables_used = None # Variables utilisées pour le VI/split

        # Enfants
        self.left = None
        self.right = None

class RLTTree:
    def __init__(self, task="classification", max_depth=10, min_samples_split=2, max_vars=5, vi_threshold=0.1, linear_split=False, random_state=42):
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_vars = max_vars
        self.vi_threshold = vi_threshold
        self.linear_split = linear_split
        self.random_state = np.random.RandomState(random_state)
        self.root = None
        self.feature_names = None

    def fit(self, X, y):
        self.feature_names = X.columns.tolist()
        self.root = self.build_node(X, y, depth=0)

    def predict(self, X):
        predictions = []
        for _, row in X.iterrows():
            predictions.append(self._predict_row(row, self.root))
        return np.array(predictions)

    def predict_path(self, X):
        paths = []
        for _, row in X.iterrows():
            path_info = []
            self._predict_row(row, self.root, path_info)
            paths.append(path_info)
        return paths

    def _predict_row(self, x, node, path_info=None):
        if node.is_leaf:
            return node.value

        if node.linear_coef is not None:
            # Linear split
            used_features = [self.feature_names[i] for i in node.variables_used]
            proj = np.dot(x[used_features], node.linear_coef)
            split_feature = f"Linear Comb ({', '.join(used_features)})";
            split_value = node.split_val
            if proj <= node.split_val:
                decision = f"Projection ({proj:.2f}) <= {node.split_val:.2f} (Left)"
                next_node = node.left
            else:
                decision = f"Projection ({proj:.2f}) > {node.split_val:.2f} (Right)"
                next_node = node.right
        else:
            # Univariate split
            split_feature = self.feature_names[node.split_var]
            split_value = node.split_val
            if x[split_feature] <= node.split_val:
                decision = f"{split_feature} ({x[split_feature]:.2f}) <= {node.split_val:.2f} (Left)"
                next_node = node.left
            else:
                decision = f"{split_feature} ({x[split_feature]:.2f}) > {node.split_val:.2f} (Right)"
                next_node = node.right

        if path_info is not None:
            path_info.append({
                'depth': node.depth,
                'feature': split_feature,
                'value': x[split_feature] if node.linear_coef is None else proj,
                'split_value': split_value,
                'decision': decision,
                'impurity_reduction': node.impurity - next_node.impurity if next_node else node.impurity
            })

        return self._predict_row(x, next_node, path_info)

    def build_node(self, X, y, depth):
        node = Node(depth=depth)
        node.impurity = self.compute_impurity(y)
        node.value = self.leaf_value(y)

        if depth >= self.max_depth or len(y) < self.min_samples_split or node.impurity == 0:
            node.is_leaf = True
            return node

        # 1. Embedded VI (ExtraTrees at node)
        vi = self.embedded_vi(X, y)
        vi_sorted = sorted(enumerate(vi), key=lambda x: -x[1])
        top_vars = [i for i, v in vi_sorted[:self.max_vars] if v >= (self.vi_threshold * max(vi))]
        if not top_vars:
            top_vars = [i for i, v in vi_sorted[:self.max_vars]]
        node.variables_used = top_vars

        # 2. Variable muting (focus only on informative vars)
        X_sub = X.iloc[:, top_vars]

        # 3. Reinforcement Learning Bandit split selection (simplified UCB/epsilon-greedy is implicit in best gain search)
        best_gain = -np.inf
        best_split = None
        best_left = None
        best_right = None
        best_linear = None

        # Univariate splits
        for var_idx in top_vars:
            col = X.iloc[:, var_idx]
            # Simplified candidate selection
            val_candidates = np.linspace(col.min(), col.max(), num=10)
            for v in val_candidates:
                left_idx = col <= v
                right_idx = col > v
                if left_idx.sum() == 0 or right_idx.sum() == 0:
                    continue

                gain = self.compute_impurity(y) - (
                    left_idx.sum() / len(y) * self.compute_impurity(y[left_idx])
                    + right_idx.sum() / len(y) * self.compute_impurity(y[right_idx]))

                if gain > best_gain:
                    best_gain = gain
                    best_split = (var_idx, v)
                    best_left = (X[left_idx], y[left_idx])
                    best_right = (X[right_idx], y[right_idx])
                    best_linear = None

        # 4. Linear combination split (optional)
        if self.linear_split and len(top_vars) >= 2:
            # Generate random coefficients
            coefs = self.random_state.randn(len(top_vars))
            # Normalize coefficients for stability
            coefs = coefs / np.linalg.norm(coefs)

            proj = np.dot(X_sub.values, coefs)
            val_candidates = np.linspace(proj.min(), proj.max(), num=10)

            for v in val_candidates:
                left_idx = proj <= v
                right_idx = proj > v
                if left_idx.sum() == 0 or right_idx.sum() == 0:
                    continue

                gain = self.compute_impurity(y) - (
                    left_idx.sum() / len(y) * self.compute_impurity(y[left_idx])
                    + right_idx.sum() / len(y) * self.compute_impurity(y[right_idx]))

                if gain > best_gain:
                    best_gain = gain
                    best_split = (None, v)
                    best_left = (X[left_idx], y[left_idx])
                    best_right = (X[right_idx], y[right_idx])
                    best_linear = coefs

        # Si pas de split, faire une feuille
        if best_gain <= 0 or best_left is None or best_right is None:
            node.is_leaf = True
            return node

        # Enregistrer le split choisi
        node.split_var, node.split_val = best_split
        node.linear_coef = best_linear

        # Pour les splits linéaires, variables_used contient les indices des caractéristiques utilisées
        if node.linear_coef is not None:
            node.variables_used = top_vars
        else:
            # Pour les splits univariés, variables_used contient l'indice de la caractéristique utilisée
            node.variables_used = [node.split_var]

        # Construire récursivement les enfants
        node.left = self.build_node(*best_left, depth=depth+1)
        node.right = self.build_node(*best_right, depth=depth+1)
        return node

    def embedded_vi(self, X, y):
        if self.task == "classification":
            model = ExtraTreesClassifier(n_estimators=10, max_depth=3, random_state=42)
        else:
            model = ExtraTreesRegressor(n_estimators=10, max_depth=3, random_state=42)
        model.fit(X, y)
        return model.feature_importances_

    def compute_impurity(self, y):
        if self.task == "classification":
            # Gini impurity
            if len(y) == 0: return 0
            probs = np.bincount(y.astype(int)) / len(y)
            return 1 - np.sum(probs ** 2)
        else:
            # Variance reduction (MSE)
            return np.var(y)

    def leaf_value(self, y):
        if self.task == "classification":
            # FIX: majority vote pour classification binaire (0/1)
            # Retourne 0 si le nœud est vide (len(y) == 0)
            if len(y) == 0: return 0
            # Le vote majoritaire est équivalent à vérifier si la moyenne est > 0.5
            return 1 if np.mean(y) > 0.5 else 0
        else:
            return np.mean(y)

# FIX: Ajout de la classe MockBART pour éviter le crash
class MockBART:
    def __init__(self):
        self.majority_class = None

    def fit(self, X, y):
        # Store the majority class from training data
        self.majority_class = int(np.round(np.mean(y)))

    def predict(self, X):
        # Return the majority class for all predictions (integer labels for classification)
        return np.full(len(X), self.majority_class, dtype=int)

**Reasoning**:
The previous step defined the `Node`, `RLTTree`, and `MockBART` classes. Now, I need to define the `RLTForest`, `RLTClassifier`, and `RLTRegressor` classes by executing the `rlt_forest_code` cell to make them available for the simulation.



In [ ]:
class RLTForest:
    def __init__(self, n_trees=10, **kwargs):
        self.n_trees = n_trees
        self.kwargs = kwargs
        self.trees = []
        self.feature_names = None

    def fit(self, X, y):
        X_df = pd.DataFrame(X, copy=True)
        y_df = pd.Series(y, index=X_df.index, copy=True)
        self.feature_names = X_df.columns.tolist()
        self.trees = []
        for i in range(self.n_trees):
            sample_idx = X_df.sample(frac=0.7, replace=True, random_state=i).index
            X_sample = X_df.loc[sample_idx]
            y_sample = y_df.loc[sample_idx]
            tree = RLTTree(**self.kwargs, random_state=i)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names)
        predictions = np.array([tree.predict(X_df) for tree in self.trees])
        if self.kwargs.get('task') == 'classification':
            # Vote majoritaire
            mode, count = Counter(predictions.flatten()).most_common(1)[0]
            return np.full(len(X), mode)
        else:
            # Moyenne
            return np.mean(predictions, axis=0)

    def predict_proba(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names)
        predictions = np.array([tree.predict(X_df) for tree in self.trees])
        # Calculer la proportion de 1s
        proba_1 = np.mean(predictions, axis=0)
        # Retourner les probabilités pour les classes 0 et 1
        return np.column_stack((1 - proba_1, proba_1))

    def get_feature_names(self):
        return self.feature_names

class RLTClassifier(RLTForest):
    def __init__(self, n_trees=10, **kwargs):
        super().__init__(n_trees=n_trees, task="classification", **kwargs)

class RLTRegressor(RLTForest):
    def __init__(self, n_trees=10, **kwargs):
        super().__init__(n_trees=n_trees, task="regression", **kwargs)